# ESP32 KWS Experiment Lab

This notebook is the orchestration layer for the rebuilt KWS training stack.

Design goals:
- experiment with multiple theories instead of one fixed recipe;
- stay close to the `micro_speech` deployment path;
- train a stronger student than the stock baseline;
- export a full-int8 `TFLite` artifact and firmware-ready C array.


In [ ]:
# Uncomment if the notebook environment is missing TensorFlow / tfmot,
# or if TensorFlow fails to import because of NumPy 2.x:
# !pip -q install "numpy<2" "tensorflow==2.15.*" "tensorflow-model-optimization>=0.7.0,<0.9" matplotlib


In [ ]:
import json
import importlib
import sys
from pathlib import Path

FORCE_SYNC_RUNTIME_FILES = True
BASE_RUNTIME_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
PROJECT_ROOT = BASE_RUNTIME_DIR / "diploma_esp32_distributed_nn"
TRAINING_ROOT = PROJECT_ROOT / "code" / "training"
FILE_PAYLOADS = json.loads('{"code/training/README.md": "# KWS Training Lab\\n\\nThis folder is the rebuilt training workspace for the speech track of the diploma project.\\n\\nThe goal is not to hard-code a single notebook experiment. The goal is to keep a clean experimentation lab that can:\\n\\n- compare multiple KWS model ideas on `Speech Commands`;\\n- stay close to the current `ESP32-S3` deployment constraints;\\n- export a full-int8 `TFLite` model plus firmware-ready C arrays;\\n- support repeated ablations and quick iteration in Jupyter.\\n\\n## Design Rules\\n\\n- Keep the notebook thin. Heavy logic lives in Python modules.\\n- Keep configs explicit and serializable.\\n- Prefer MCU-friendly building blocks and simple TFLite op sets.\\n- Make every experiment reproducible through named recipes.\\n\\n## Main Files\\n\\n- `build_kws_experiments_notebook.py`: generates the notebook.\\n- `kws_advanced/config.py`: experiment dataclasses and defaults.\\n- `kws_advanced/experiments.py`: named experiment recipes for repeated tests.\\n- `kws_advanced/data.py`: dataset indexing, tf.data input pipeline, and augmentation.\\n- `kws_advanced/features.py`: feature extraction and feature-level augmentation.\\n- `kws_advanced/models.py`: teacher/student architectures and model metrics.\\n- `kws_advanced/distillation.py`: compile helpers, callbacks, and distillation model.\\n- `kws_advanced/export.py`: int8 export, metadata dump, and C-array generation.\\n- `kws_advanced/reporting.py`: run manifests, history plots, metrics dumps, and markdown summaries.\\n\\n## Recommended Workflow\\n\\n1. Generate or refresh the notebook:\\n\\n```powershell\\npython code/training/build_kws_experiments_notebook.py\\n```\\n\\n2. Open `code/training/kws_experiments_esp32.ipynb`.\\n\\n3. Pick an experiment recipe.\\n\\n4. Train teacher -> distill student -> optional QAT.\\n\\n5. Export the final student model and record the resulting metrics and artifacts.\\n\\n6. Read the saved run bundle under `code/training/runs/<experiment-tag>/`. Each run now writes:\\n\\n- `run_state.json`: machine-readable experiment state.\\n- `run_summary.md`: human-readable summary with config, stages, metrics, and artifacts.\\n- `teacher|student|qat_history.json`: epoch histories.\\n- `teacher|student|qat_metrics.json`: final evaluation metrics.\\n- `teacher|student|qat_loss.png` and `*_metrics.png`: saved plots.\\n- `teacher|student|qat/epoch_log.csv`: per-epoch logs written during training.\\n- `frontend_runtime.json`: exact `microfrontend` vs fallback runtime info.\\n- `<experiment-tag>_bundle.zip`: final archive of the whole run bundle for export from Colab.\\n\\nThe default next-iteration notebook recipe is now `distilled_student_v3_bcresnet_teacher_seq`. The current conclusion is that the richer-frontend `KWT` teacher is excellent as a teacher benchmark, but the mainline student path should currently stay on a more stable same-frontend `BC-ResNet -> compact student` distillation setup.\\n\\nThe notebook now also supports teacher reuse from Google Drive. By default it will:\\n\\n- mount Drive when available;\\n- look under `/content/drive/MyDrive/diploma_esp32_teacher_saves/` for the latest `teacher_best.keras` matching `experiment.teacher_reuse_tag` when it is set, otherwise the current `experiment.tag`;\\n- load that checkpoint and skip teacher training if found.\\n\\nYou can still override this manually inside the notebook by editing:\\n\\n- `REUSE_TEACHER_CHECKPOINT`\\n- `TEACHER_CHECKPOINT_PATH`\\n- `AUTO_FIND_LATEST_TEACHER_CHECKPOINT`\\n- `TEACHER_SEARCH_ROOT`\\n\\nThe training stack now also supports split teacher/student frontends inside one run:\\n\\n- teacher datasets can use a richer teacher-only frontend;\\n- student datasets stay on the deployment-oriented frontend;\\n- distillation datasets carry both feature views so KD can run without forcing the student off the firmware path.\\n\\n## Environment Note\\n\\nIf TensorFlow fails to import because of `NumPy 2.x`, use a compatible stack such as:\\n\\n```powershell\\npip install -r code/training/requirements-kws.txt\\n```\\n\\nIf you run the notebook from `VS Code` while connected to a `Colab` kernel, the runtime does not see your local `C:\\\\...` repo automatically. The notebook bootstrap solves this by writing the required training files into a runtime workspace under `/content/diploma_esp32_distributed_nn/`. The current notebook version force-syncs those embedded sources on every launch so the runtime does not keep stale modules from older notebook revisions.\\n\\nIf dataset download fails because of SSL or network policy, set `SPEECH_COMMANDS_DATASET_ROOT` to an already extracted `speech_commands_v0.02` folder, or set `SPEECH_COMMANDS_ARCHIVE_PATH` to a local `speech_commands_v0.02.tar.gz` file in the notebook override cell.\\n", "code/training/requirements-kws.txt": "numpy<2\\ntensorflow==2.15.*\\ntensorflow-model-optimization>=0.7.0,<0.9\\nmatplotlib>=3.8,<4\\n", "code/training/kws_advanced/__init__.py": "from .config import ExperimentConfig, make_experiment\\nfrom .experiments import build_recipe_book\\n\\n__all__ = [\\n    \\"ExperimentConfig\\",\\n    \\"build_recipe_book\\",\\n    \\"make_experiment\\",\\n]\\n", "code/training/kws_advanced/config.py": "from __future__ import annotations\\n\\nfrom dataclasses import asdict, dataclass, field\\nfrom typing import Any\\n\\n\\nVOCABULARY_PRESETS: dict[str, list[str]] = {\\n    \\"kws4\\": [\\"yes\\", \\"no\\"],\\n    \\"kws12\\": [\\"yes\\", \\"no\\", \\"up\\", \\"down\\", \\"left\\", \\"right\\", \\"on\\", \\"off\\", \\"stop\\", \\"go\\"],\\n    \\"kws20\\": [\\n        \\"yes\\",\\n        \\"no\\",\\n        \\"up\\",\\n        \\"down\\",\\n        \\"left\\",\\n        \\"right\\",\\n        \\"on\\",\\n        \\"off\\",\\n        \\"stop\\",\\n        \\"go\\",\\n        \\"zero\\",\\n        \\"one\\",\\n        \\"two\\",\\n        \\"three\\",\\n        \\"four\\",\\n        \\"five\\",\\n        \\"six\\",\\n        \\"seven\\",\\n        \\"eight\\",\\n        \\"nine\\",\\n    ],\\n}\\n\\n\\n@dataclass(frozen=True)\\nclass FrontendConfig:\\n    sample_rate: int = 16_000\\n    clip_ms: int = 1_000\\n    frame_length_ms: int = 30\\n    frame_step_ms: int = 20\\n    fft_length: int = 512\\n    num_channels: int = 40\\n    lower_band_limit: float = 125.0\\n    upper_band_limit: float = 7_500.0\\n    enable_exact_microfrontend: bool = True\\n    use_pcen_fallback: bool = True\\n    smoothing_bits: int = 10\\n    even_smoothing: float = 0.025\\n    odd_smoothing: float = 0.06\\n    min_signal_remaining: float = 0.05\\n    pcan_strength: float = 0.95\\n    pcan_offset: float = 80.0\\n    gain_bits: int = 21\\n    scale_shift: int = 6\\n    fallback_feature_clip: float = 8.0\\n    pcen_alpha: float = 0.98\\n    pcen_delta: float = 2.0\\n    pcen_root: float = 0.5\\n    pcen_floor: float = 1e-6\\n    pcen_smoothing_coef: float = 0.04\\n\\n    @property\\n    def desired_samples(self) -> int:\\n        return (self.sample_rate * self.clip_ms) // 1000\\n\\n    @property\\n    def frame_length_samples(self) -> int:\\n        return (self.sample_rate * self.frame_length_ms) // 1000\\n\\n    @property\\n    def frame_step_samples(self) -> int:\\n        return (self.sample_rate * self.frame_step_ms) // 1000\\n\\n    @property\\n    def frame_count(self) -> int:\\n        usable = self.desired_samples - self.frame_length_samples\\n        return 1 + max(0, usable // self.frame_step_samples)\\n\\n    @property\\n    def feature_shape(self) -> tuple[int, int, int]:\\n        return (self.frame_count, self.num_channels, 1)\\n\\n\\n@dataclass(frozen=True)\\nclass DatasetConfig:\\n    data_dir: str = \\"data\\"\\n    dataset_root_override: str = \\"\\"\\n    archive_path_override: str = \\"\\"\\n    batch_size: int = 96\\n    shuffle_buffer: int = 6_000\\n    seed: int = 42\\n    unknown_ratio: float = 1.0\\n    silence_ratio: float = 0.12\\n    train_fraction: float = 1.0\\n    eval_fraction: float = 1.0\\n    cache_validation: bool = True\\n    time_shift_ms: int = 100\\n    gain_min: float = 0.8\\n    gain_max: float = 1.2\\n    noise_stddev: float = 0.004\\n    mixup_alpha: float = 0.1\\n    specaugment_prob: float = 0.5\\n    time_mask_max: int = 4\\n    freq_mask_max: int = 3\\n\\n\\n@dataclass(frozen=True)\\nclass ModelConfig:\\n    teacher_name: str = \\"teacher_factorized_dscnn\\"\\n    student_name: str = \\"student_factorized_dscnn_v2\\"\\n    teacher_width: float = 1.0\\n    student_width: float = 0.875\\n    teacher_dropout: float = 0.20\\n    student_dropout: float = 0.08\\n    teacher_label_smoothing: float = 0.01\\n    label_smoothing: float = 0.02\\n\\n\\n@dataclass(frozen=True)\\nclass TrainConfig:\\n    teacher_epochs: int = 14\\n    student_pretrain_epochs: int = 0\\n    student_epochs: int = 18\\n    qat_epochs: int = 4\\n    student_polish_epochs: int = 0\\n    teacher_warmup_epochs: int = 0\\n    teacher_lr: float = 1e-3\\n    student_lr: float = 9e-4\\n    qat_lr: float = 1e-4\\n    ce_loss_weight: float = 0.2\\n    kd_loss_weight: float = 0.8\\n    distill_temperature: float = 5.0\\n    distillation_mode: str = \\"logits\\"\\n    hint_warmup_epochs: int = 0\\n    hint_warmup_ce_weight: float = 0.4\\n    hint_warmup_hint_weight: float = 0.6\\n    hint_loss_weight: float = 0.0\\n    attention_loss_weight: float = 0.0\\n    similarity_loss_weight: float = 0.0\\n    hint_layer_pair: tuple[str, str] = (\\"stage3_block1_out\\", \\"stage3_block2_out\\")\\n    attention_layer_pairs: tuple[tuple[str, str], ...] = ()\\n    similarity_layer_pair: tuple[str, str] = (\\"\\", \\"\\")\\n    use_qat: bool = False\\n    teacher_early_stopping_patience: int = 4\\n    teacher_early_stopping_min_delta: float = 0.0\\n    early_stopping_patience: int = 4\\n    early_stopping_min_delta: float = 0.0\\n    top_k: int = 3\\n    optimizer_name: str = \\"adamw\\"\\n    weight_decay: float = 1.5e-4\\n    clipnorm: float = 1.0\\n\\n    @property\\n    def uses_distillation(self) -> bool:\\n        return any(\\n            weight > 0.0\\n            for weight in (\\n                self.kd_loss_weight,\\n                self.hint_loss_weight,\\n                self.attention_loss_weight,\\n                self.similarity_loss_weight,\\n            )\\n        )\\n\\n\\n@dataclass(frozen=True)\\nclass ExportConfig:\\n    artifacts_dir: str = \\"code/training/artifacts\\"\\n    model_stem: str = \\"kws_esp32_advanced\\"\\n    representative_batches: int = 200\\n    export_c_array: bool = True\\n\\n\\n@dataclass(frozen=True)\\nclass ExperimentConfig:\\n    tag: str\\n    vocabulary_preset: str\\n    teacher_reuse_tag: str = \\"\\"\\n    frontend: FrontendConfig = field(default_factory=FrontendConfig)\\n    teacher_frontend: FrontendConfig | None = None\\n    student_frontend: FrontendConfig | None = None\\n    dataset: DatasetConfig = field(default_factory=DatasetConfig)\\n    model: ModelConfig = field(default_factory=ModelConfig)\\n    train: TrainConfig = field(default_factory=TrainConfig)\\n    export: ExportConfig = field(default_factory=ExportConfig)\\n\\n    @property\\n    def commands(self) -> list[str]:\\n        if self.vocabulary_preset not in VOCABULARY_PRESETS:\\n            raise KeyError(f\\"Unknown vocabulary preset: {self.vocabulary_preset}\\")\\n        return list(VOCABULARY_PRESETS[self.vocabulary_preset])\\n\\n    @property\\n    def all_labels(self) -> list[str]:\\n        return [\\"silence\\", \\"unknown\\", *self.commands]\\n\\n    @property\\n    def label_to_index(self) -> dict[str, int]:\\n        return {label: index for index, label in enumerate(self.all_labels)}\\n\\n    @property\\n    def num_labels(self) -> int:\\n        return len(self.all_labels)\\n\\n    @property\\n    def teacher_frontend_config(self) -> FrontendConfig:\\n        return self.teacher_frontend or self.frontend\\n\\n    @property\\n    def student_frontend_config(self) -> FrontendConfig:\\n        return self.student_frontend or self.frontend\\n\\n    @property\\n    def teacher_feature_shape(self) -> tuple[int, int, int]:\\n        return self.teacher_frontend_config.feature_shape\\n\\n    @property\\n    def student_feature_shape(self) -> tuple[int, int, int]:\\n        return self.student_frontend_config.feature_shape\\n\\n    @property\\n    def feature_shape(self) -> tuple[int, int, int]:\\n        return self.student_feature_shape\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        data = asdict(self)\\n        data[\\"commands\\"] = self.commands\\n        data[\\"all_labels\\"] = self.all_labels\\n        data[\\"feature_shape\\"] = list(self.feature_shape)\\n        data[\\"teacher_feature_shape\\"] = list(self.teacher_feature_shape)\\n        data[\\"student_feature_shape\\"] = list(self.student_feature_shape)\\n        data[\\"num_labels\\"] = self.num_labels\\n        return data\\n\\n\\ndef make_experiment(\\n    tag: str = \\"kws12_iterlab_v1\\",\\n    vocabulary_preset: str = \\"kws12\\",\\n) -> ExperimentConfig:\\n    return ExperimentConfig(\\n        tag=tag,\\n        vocabulary_preset=vocabulary_preset,\\n    )\\n", "code/training/kws_advanced/experiments.py": "from __future__ import annotations\\n\\nfrom dataclasses import replace\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef build_recipe_book(base: ExperimentConfig) -> dict[str, ExperimentConfig]:\\n    baseline = replace(\\n        base,\\n        tag=f\\"{base.tag}_baseline\\",\\n        dataset=replace(\\n            base.dataset,\\n            batch_size=128,\\n            mixup_alpha=0.0,\\n            specaugment_prob=0.3,\\n            time_mask_max=2,\\n            freq_mask_max=2,\\n        ),\\n        model=replace(\\n            base.model,\\n            teacher_name=\\"baseline_cnn\\",\\n            student_name=\\"baseline_cnn\\",\\n            teacher_width=1.0,\\n            student_width=1.0,\\n            teacher_dropout=0.15,\\n            student_dropout=0.15,\\n            teacher_label_smoothing=0.01,\\n        ),\\n        train=replace(\\n            base.train,\\n            ce_loss_weight=1.0,\\n            kd_loss_weight=0.0,\\n            use_qat=False,\\n            teacher_epochs=12,\\n            student_epochs=12,\\n            teacher_early_stopping_patience=4,\\n            teacher_early_stopping_min_delta=0.0,\\n            optimizer_name=\\"adam\\",\\n            weight_decay=0.0,\\n        ),\\n    )\\n\\n    fast_student = replace(\\n        base,\\n        tag=f\\"{base.tag}_fast_student\\",\\n        model=replace(\\n            base.model,\\n            student_name=\\"student_factorized_dscnn\\",\\n            student_width=0.65,\\n            student_dropout=0.08,\\n        ),\\n        train=replace(\\n            base.train,\\n            ce_loss_weight=1.0,\\n            kd_loss_weight=0.0,\\n            use_qat=False,\\n            student_epochs=14,\\n        ),\\n    )\\n\\n    distilled_student = replace(\\n        base,\\n        tag=f\\"{base.tag}_distilled_v1ref\\",\\n        dataset=replace(\\n            base.dataset,\\n            batch_size=128,\\n            noise_stddev=0.005,\\n            mixup_alpha=0.2,\\n            specaugment_prob=0.7,\\n            time_mask_max=5,\\n            freq_mask_max=4,\\n        ),\\n        model=replace(\\n            base.model,\\n            teacher_name=\\"teacher_factorized_dscnn\\",\\n            student_name=\\"student_factorized_dscnn\\",\\n            teacher_width=1.0,\\n            student_width=0.75,\\n            student_dropout=0.12,\\n            teacher_label_smoothing=0.01,\\n            label_smoothing=0.05,\\n        ),\\n        train=replace(\\n            base.train,\\n            teacher_epochs=18,\\n            student_epochs=24,\\n            ce_loss_weight=0.35,\\n            kd_loss_weight=0.65,\\n            distill_temperature=4.0,\\n            distillation_mode=\\"logits\\",\\n            use_qat=False,\\n            teacher_early_stopping_patience=6,\\n            teacher_early_stopping_min_delta=0.0,\\n            early_stopping_patience=6,\\n            optimizer_name=\\"adam\\",\\n            weight_decay=0.0,\\n        ),\\n    )\\n\\n    distilled_qat = replace(\\n        distilled_student,\\n        tag=f\\"{base.tag}_distilled_qat_v1ref\\",\\n        train=replace(\\n            distilled_student.train,\\n            use_qat=True,\\n            qat_epochs=6,\\n        ),\\n    )\\n\\n    distilled_student_v2 = replace(\\n        base,\\n        tag=f\\"{base.tag}_distilled_v2\\",\\n        model=replace(\\n            base.model,\\n            teacher_name=\\"teacher_factorized_dscnn\\",\\n            student_name=\\"student_factorized_dscnn_v2\\",\\n            teacher_width=1.0,\\n            student_width=0.875,\\n            student_dropout=0.08,\\n            teacher_label_smoothing=0.01,\\n            label_smoothing=0.02,\\n        ),\\n        train=replace(\\n            base.train,\\n            use_qat=False,\\n            ce_loss_weight=0.2,\\n            kd_loss_weight=0.8,\\n            distill_temperature=5.0,\\n            distillation_mode=\\"logits\\",\\n            teacher_epochs=14,\\n            student_epochs=18,\\n            teacher_early_stopping_patience=4,\\n            teacher_early_stopping_min_delta=0.0,\\n            early_stopping_patience=4,\\n            optimizer_name=\\"adamw\\",\\n            weight_decay=1.5e-4,\\n        ),\\n    )\\n\\n    distilled_student_v3 = replace(\\n        distilled_student_v2,\\n        tag=f\\"{base.tag}_distilled_v3\\",\\n        train=replace(\\n            distilled_student_v2.train,\\n            teacher_epochs=10,\\n            student_epochs=10,\\n            student_polish_epochs=1,\\n            ce_loss_weight=0.28,\\n            kd_loss_weight=0.5,\\n            distill_temperature=6.0,\\n            distillation_mode=\\"multilevel\\",\\n            hint_warmup_epochs=2,\\n            hint_warmup_ce_weight=0.35,\\n            hint_warmup_hint_weight=0.65,\\n            hint_loss_weight=0.14,\\n            attention_loss_weight=0.08,\\n            similarity_loss_weight=0.04,\\n            hint_layer_pair=(\\"stage3_block1_out\\", \\"stage3_block2_out\\"),\\n            attention_layer_pairs=(\\n                (\\"stage2_block2_out\\", \\"stage2_block2_out\\"),\\n                (\\"stage3_block1_out\\", \\"stage3_block2_out\\"),\\n            ),\\n            similarity_layer_pair=(\\"stage3_block1_out\\", \\"stage3_block2_out\\"),\\n            teacher_early_stopping_patience=4,\\n            teacher_early_stopping_min_delta=0.0,\\n            early_stopping_patience=2,\\n            early_stopping_min_delta=0.002,\\n        ),\\n    )\\n\\n    distilled_student_v3_teacher_plus = replace(\\n        distilled_student_v3,\\n        tag=f\\"{base.tag}_distilled_v3_teacher_plus\\",\\n        model=replace(\\n            distilled_student_v3.model,\\n            teacher_name=\\"teacher_factorized_dscnn_xl\\",\\n            teacher_width=1.0,\\n            teacher_dropout=0.15,\\n            teacher_label_smoothing=0.01,\\n        ),\\n        train=replace(\\n            distilled_student_v3.train,\\n            teacher_epochs=16,\\n            teacher_early_stopping_patience=5,\\n            teacher_early_stopping_min_delta=0.0,\\n        ),\\n    )\\n\\n    distilled_student_v3_bcresnet_teacher = replace(\\n        distilled_student_v3,\\n        tag=f\\"{base.tag}_distilled_v3_bcresnet_teacher\\",\\n        model=replace(\\n            distilled_student_v3.model,\\n            teacher_name=\\"teacher_bcresnet\\",\\n            teacher_width=1.0,\\n            teacher_dropout=0.0,\\n            teacher_label_smoothing=0.0,\\n        ),\\n        train=replace(\\n            distilled_student_v3.train,\\n            teacher_epochs=20,\\n            teacher_early_stopping_patience=6,\\n            teacher_early_stopping_min_delta=0.0,\\n        ),\\n    )\\n\\n    distilled_student_v3_bcresnet_teacher_seq = replace(\\n        distilled_student_v3_bcresnet_teacher,\\n        tag=f\\"{base.tag}_distilled_v3_bcresnet_teacher_seq\\",\\n        dataset=replace(\\n            distilled_student_v3_bcresnet_teacher.dataset,\\n            batch_size=96,\\n            mixup_alpha=0.08,\\n            specaugment_prob=0.45,\\n            time_mask_max=5,\\n            freq_mask_max=4,\\n        ),\\n        train=replace(\\n            distilled_student_v3_bcresnet_teacher.train,\\n            student_pretrain_epochs=6,\\n            student_epochs=10,\\n            student_polish_epochs=1,\\n            ce_loss_weight=0.45,\\n            kd_loss_weight=0.55,\\n            distill_temperature=4.5,\\n            distillation_mode=\\"logits\\",\\n            hint_warmup_epochs=0,\\n            hint_warmup_ce_weight=0.0,\\n            hint_warmup_hint_weight=0.0,\\n            hint_loss_weight=0.0,\\n            attention_loss_weight=0.0,\\n            similarity_loss_weight=0.0,\\n            hint_layer_pair=(\\"\\", \\"\\"),\\n            attention_layer_pairs=(),\\n            similarity_layer_pair=(\\"\\", \\"\\"),\\n            early_stopping_patience=4,\\n            early_stopping_min_delta=0.001,\\n        ),\\n    )\\n\\n    distilled_student_v3_kwt_teacher = replace(\\n        distilled_student_v3,\\n        tag=f\\"{base.tag}_distilled_v3_kwt_teacher\\",\\n        dataset=replace(\\n            distilled_student_v3.dataset,\\n            batch_size=128,\\n            mixup_alpha=0.0,\\n            specaugment_prob=0.7,\\n            time_mask_max=6,\\n            freq_mask_max=4,\\n        ),\\n        model=replace(\\n            distilled_student_v3.model,\\n            teacher_name=\\"teacher_kwt\\",\\n            teacher_width=1.0,\\n            teacher_dropout=0.0,\\n            teacher_label_smoothing=0.1,\\n        ),\\n        train=replace(\\n            distilled_student_v3.train,\\n            teacher_epochs=40,\\n            teacher_warmup_epochs=8,\\n            teacher_lr=5e-4,\\n            student_epochs=12,\\n            student_polish_epochs=1,\\n            ce_loss_weight=0.25,\\n            kd_loss_weight=0.75,\\n            distill_temperature=6.0,\\n            distillation_mode=\\"logits\\",\\n            hint_warmup_epochs=0,\\n            hint_warmup_ce_weight=0.0,\\n            hint_warmup_hint_weight=0.0,\\n            hint_loss_weight=0.0,\\n            attention_loss_weight=0.0,\\n            similarity_loss_weight=0.0,\\n            hint_layer_pair=(\\"\\", \\"\\"),\\n            attention_layer_pairs=(),\\n            similarity_layer_pair=(\\"\\", \\"\\"),\\n            teacher_early_stopping_patience=8,\\n            teacher_early_stopping_min_delta=0.0,\\n            early_stopping_patience=4,\\n            early_stopping_min_delta=0.001,\\n            optimizer_name=\\"adamw\\",\\n            weight_decay=5e-4,\\n        ),\\n    )\\n\\n    distilled_student_v3_kwt_rich_teacher = replace(\\n        distilled_student_v3_kwt_teacher,\\n        tag=f\\"{base.tag}_distilled_v3_kwt_rich_teacher\\",\\n        teacher_frontend=replace(\\n            distilled_student_v3.frontend,\\n            enable_exact_microfrontend=False,\\n            use_pcen_fallback=False,\\n            frame_length_ms=25,\\n            frame_step_ms=10,\\n            num_channels=64,\\n            lower_band_limit=60.0,\\n            upper_band_limit=7_800.0,\\n            fallback_feature_clip=10.0,\\n        ),\\n        student_frontend=base.frontend,\\n        dataset=replace(\\n            distilled_student_v3_kwt_teacher.dataset,\\n            batch_size=96,\\n            specaugment_prob=0.5,\\n            time_mask_max=8,\\n            freq_mask_max=6,\\n        ),\\n        model=replace(\\n            distilled_student_v3_kwt_teacher.model,\\n            teacher_label_smoothing=0.05,\\n        ),\\n        train=replace(\\n            distilled_student_v3_kwt_teacher.train,\\n            teacher_epochs=36,\\n            teacher_warmup_epochs=8,\\n            teacher_lr=4e-4,\\n            teacher_early_stopping_patience=8,\\n            teacher_early_stopping_min_delta=0.0,\\n        ),\\n    )\\n\\n    distilled_student_v3_kwt_rich_teacher_seq = replace(\\n        distilled_student_v3_kwt_rich_teacher,\\n        tag=f\\"{base.tag}_distilled_v3_kwt_rich_teacher_seq\\",\\n        teacher_reuse_tag=distilled_student_v3_kwt_rich_teacher.tag,\\n        dataset=replace(\\n            distilled_student_v3_kwt_rich_teacher.dataset,\\n            mixup_alpha=0.05,\\n            specaugment_prob=0.4,\\n            time_mask_max=6,\\n            freq_mask_max=4,\\n        ),\\n        train=replace(\\n            distilled_student_v3_kwt_rich_teacher.train,\\n            student_pretrain_epochs=6,\\n            student_epochs=8,\\n            student_polish_epochs=1,\\n            ce_loss_weight=0.45,\\n            kd_loss_weight=0.55,\\n            distill_temperature=4.5,\\n            early_stopping_patience=3,\\n            early_stopping_min_delta=0.001,\\n        ),\\n    )\\n\\n    assistant_bcresnet_from_kwt_rich_teacher = replace(\\n        distilled_student_v3_kwt_rich_teacher,\\n        tag=f\\"{base.tag}_assistant_bcresnet_from_kwt_rich_teacher\\",\\n        teacher_reuse_tag=distilled_student_v3_kwt_rich_teacher.tag,\\n        dataset=replace(\\n            distilled_student_v3_kwt_rich_teacher.dataset,\\n            batch_size=96,\\n            mixup_alpha=0.05,\\n            specaugment_prob=0.35,\\n            time_mask_max=5,\\n            freq_mask_max=4,\\n        ),\\n        model=replace(\\n            distilled_student_v3_kwt_rich_teacher.model,\\n            student_name=\\"teacher_bcresnet\\",\\n            student_width=1.0,\\n            student_dropout=0.0,\\n            label_smoothing=0.01,\\n        ),\\n        train=replace(\\n            distilled_student_v3_kwt_rich_teacher.train,\\n            student_pretrain_epochs=4,\\n            student_epochs=10,\\n            student_polish_epochs=1,\\n            ce_loss_weight=0.40,\\n            kd_loss_weight=0.60,\\n            distill_temperature=4.5,\\n            early_stopping_patience=4,\\n            early_stopping_min_delta=0.001,\\n        ),\\n    )\\n\\n    distilled_qat_v2 = replace(\\n        distilled_student_v2,\\n        tag=f\\"{base.tag}_distilled_qat_v2\\",\\n        train=replace(\\n            distilled_student_v2.train,\\n            use_qat=True,\\n            qat_epochs=4,\\n        ),\\n    )\\n\\n    kws20_probe = replace(\\n        base,\\n        tag=\\"kws20_probe\\",\\n        vocabulary_preset=\\"kws20\\",\\n        dataset=replace(\\n            base.dataset,\\n            batch_size=96,\\n            mixup_alpha=0.15,\\n        ),\\n        train=replace(\\n            base.train,\\n            teacher_epochs=22,\\n            student_epochs=26,\\n        ),\\n    )\\n\\n    return {\\n        \\"baseline\\": baseline,\\n        \\"fast_student\\": fast_student,\\n        \\"distilled_student\\": distilled_student,\\n        \\"distilled_qat\\": distilled_qat,\\n        \\"distilled_student_v2\\": distilled_student_v2,\\n        \\"distilled_student_v3\\": distilled_student_v3,\\n        \\"distilled_student_v3_teacher_plus\\": distilled_student_v3_teacher_plus,\\n        \\"distilled_student_v3_bcresnet_teacher\\": distilled_student_v3_bcresnet_teacher,\\n        \\"distilled_student_v3_bcresnet_teacher_seq\\": distilled_student_v3_bcresnet_teacher_seq,\\n        \\"distilled_student_v3_kwt_teacher\\": distilled_student_v3_kwt_teacher,\\n        \\"distilled_student_v3_kwt_rich_teacher\\": distilled_student_v3_kwt_rich_teacher,\\n        \\"distilled_student_v3_kwt_rich_teacher_seq\\": distilled_student_v3_kwt_rich_teacher_seq,\\n        \\"assistant_bcresnet_from_kwt_rich_teacher\\": assistant_bcresnet_from_kwt_rich_teacher,\\n        \\"distilled_qat_v2\\": distilled_qat_v2,\\n        \\"kws20_probe\\": kws20_probe,\\n    }\\n\\n\\ndef recipe_names(base: ExperimentConfig) -> list[str]:\\n    return list(build_recipe_book(base).keys())\\n", "code/training/kws_advanced/features.py": "from __future__ import annotations\\n\\nfrom functools import lru_cache\\n\\nimport tensorflow as tf\\n\\nfrom .config import FrontendConfig\\n\\n\\ndef pad_or_trim_waveform(waveform: tf.Tensor, config: FrontendConfig) -> tf.Tensor:\\n    waveform = tf.cast(waveform, tf.float32)\\n    waveform = waveform[: config.desired_samples]\\n    padding = tf.maximum(0, config.desired_samples - tf.shape(waveform)[0])\\n    waveform = tf.pad(waveform, [[0, padding]])\\n    waveform.set_shape([config.desired_samples])\\n    return waveform\\n\\n\\n@lru_cache(maxsize=1)\\ndef _microfrontend_module():\\n    try:\\n        from tensorflow.lite.experimental.microfrontend.python.ops import (  # type: ignore\\n            audio_microfrontend_op,\\n        )\\n    except Exception:\\n        return None\\n    return audio_microfrontend_op\\n\\n\\ndef describe_frontend_runtime(config: FrontendConfig) -> dict[str, object]:\\n    module_available = _microfrontend_module() is not None\\n    using_exact = bool(module_available and config.enable_exact_microfrontend)\\n    return {\\n        \\"feature_shape\\": list(config.feature_shape),\\n        \\"enable_exact_microfrontend\\": config.enable_exact_microfrontend,\\n        \\"microfrontend_module_available\\": module_available,\\n        \\"using_exact_microfrontend\\": using_exact,\\n        \\"fallback_enabled\\": config.use_pcen_fallback,\\n        \\"fallback_feature_clip\\": config.fallback_feature_clip,\\n        \\"frame_length_ms\\": config.frame_length_ms,\\n        \\"frame_step_ms\\": config.frame_step_ms,\\n        \\"num_channels\\": config.num_channels,\\n    }\\n\\n\\ndef _waveform_to_int16(waveform: tf.Tensor) -> tf.Tensor:\\n    waveform = tf.clip_by_value(tf.cast(waveform, tf.float32), -1.0, 1.0)\\n    return tf.cast(tf.round(waveform * 32767.0), tf.int16)\\n\\n\\ndef _exact_microfrontend_features(\\n    waveform: tf.Tensor,\\n    config: FrontendConfig,\\n) -> tf.Tensor | None:\\n    module = _microfrontend_module()\\n    if module is None or not config.enable_exact_microfrontend:\\n        return None\\n\\n    audio = _waveform_to_int16(waveform)\\n    common_kwargs = {\\n        \\"sample_rate\\": config.sample_rate,\\n        \\"num_channels\\": config.num_channels,\\n        \\"lower_band_limit\\": config.lower_band_limit,\\n        \\"upper_band_limit\\": config.upper_band_limit,\\n        \\"smoothing_bits\\": config.smoothing_bits,\\n        \\"even_smoothing\\": config.even_smoothing,\\n        \\"odd_smoothing\\": config.odd_smoothing,\\n        \\"min_signal_remaining\\": config.min_signal_remaining,\\n        \\"enable_pcan\\": True,\\n        \\"pcan_strength\\": config.pcan_strength,\\n        \\"pcan_offset\\": config.pcan_offset,\\n        \\"gain_bits\\": config.gain_bits,\\n        \\"enable_log\\": True,\\n        \\"scale_shift\\": config.scale_shift,\\n        \\"left_context\\": 0,\\n        \\"right_context\\": 0,\\n        \\"frame_stride\\": 1,\\n        \\"zero_padding\\": False,\\n    }\\n    signature_variants = (\\n        {\\n            \\"window_size\\": config.frame_length_ms,\\n            \\"window_step\\": config.frame_step_ms,\\n        },\\n        {\\n            \\"window_size_ms\\": config.frame_length_ms,\\n            \\"window_step_ms\\": config.frame_step_ms,\\n        },\\n    )\\n    for variant in signature_variants:\\n        try:\\n            features = module.audio_microfrontend(audio, **common_kwargs, **variant)\\n            return tf.cast(features, tf.float32)\\n        except TypeError:\\n            continue\\n    return None\\n\\n\\ndef _mel_weight_matrix(config_key: tuple[int, int, int, float, float]) -> tf.Tensor:\\n    sample_rate, fft_length, num_channels, lower_band, upper_band = config_key\\n    return tf.signal.linear_to_mel_weight_matrix(\\n        num_mel_bins=num_channels,\\n        num_spectrogram_bins=(fft_length // 2) + 1,\\n        sample_rate=sample_rate,\\n        lower_edge_hertz=lower_band,\\n        upper_edge_hertz=upper_band,\\n    )\\n\\n\\ndef _ema_over_time(values: tf.Tensor, coefficient: float) -> tf.Tensor:\\n    values = tf.cast(values, tf.float32)\\n    coefficient = tf.cast(coefficient, tf.float32)\\n\\n    def step(prev: tf.Tensor, current: tf.Tensor) -> tf.Tensor:\\n        return (coefficient * prev) + ((1.0 - coefficient) * current)\\n\\n    first = values[0]\\n    rest = values[1:]\\n    smoothed_rest = tf.scan(step, rest, initializer=first)\\n    return tf.concat([first[tf.newaxis, :], smoothed_rest], axis=0)\\n\\n\\ndef _fallback_log_mel_or_pcen(\\n    waveform: tf.Tensor,\\n    config: FrontendConfig,\\n) -> tf.Tensor:\\n    stft = tf.signal.stft(\\n        waveform,\\n        frame_length=config.frame_length_samples,\\n        frame_step=config.frame_step_samples,\\n        fft_length=config.fft_length,\\n        window_fn=tf.signal.hann_window,\\n        pad_end=False,\\n    )\\n    power_spectrogram = tf.math.square(tf.abs(stft))\\n    mel_matrix = _mel_weight_matrix(\\n        (\\n            config.sample_rate,\\n            config.fft_length,\\n            config.num_channels,\\n            config.lower_band_limit,\\n            config.upper_band_limit,\\n        )\\n    )\\n    mel = tf.tensordot(power_spectrogram, mel_matrix, axes=1)\\n    mel = tf.cast(mel, tf.float32)\\n    mel.set_shape([None, config.num_channels])\\n    mel = tf.maximum(mel, config.pcen_floor)\\n\\n    if config.use_pcen_fallback:\\n        smoother = _ema_over_time(mel, coefficient=config.pcen_smoothing_coef)\\n        normalized = mel / tf.pow(config.pcen_floor + smoother, config.pcen_alpha)\\n        pcen = tf.pow(normalized + config.pcen_delta, config.pcen_root) - (\\n            config.pcen_delta ** config.pcen_root\\n        )\\n        return tf.math.log1p(tf.maximum(pcen, 0.0))\\n\\n    return tf.math.log1p(10.0 * mel)\\n\\n\\ndef _quantize_exact_microfrontend(features: tf.Tensor) -> tf.Tensor:\\n    value_divisor = 25.6 * 26.0\\n    scaled = tf.round((features * 256.0) / value_divisor) - 128.0\\n    return tf.clip_by_value(scaled, -128.0, 127.0)\\n\\n\\ndef _quantize_fallback_features(\\n    features: tf.Tensor,\\n    config: FrontendConfig,\\n) -> tf.Tensor:\\n    clipped = tf.clip_by_value(features, 0.0, config.fallback_feature_clip)\\n    scaled = (clipped / config.fallback_feature_clip) * 255.0 - 128.0\\n    return tf.clip_by_value(tf.round(scaled), -128.0, 127.0)\\n\\n\\ndef _fit_feature_frames(features: tf.Tensor, config: FrontendConfig) -> tf.Tensor:\\n    features = features[: config.frame_count]\\n    pad_frames = tf.maximum(0, config.frame_count - tf.shape(features)[0])\\n    features = tf.pad(features, [[0, pad_frames], [0, 0]])\\n    features.set_shape([config.frame_count, config.num_channels])\\n    return features\\n\\n\\ndef extract_feature_map(\\n    waveform: tf.Tensor,\\n    config: FrontendConfig,\\n) -> tf.Tensor:\\n    waveform = pad_or_trim_waveform(waveform, config)\\n    exact = _exact_microfrontend_features(waveform, config)\\n    if exact is not None:\\n        features = _quantize_exact_microfrontend(exact)\\n    else:\\n        fallback = _fallback_log_mel_or_pcen(waveform, config)\\n        features = _quantize_fallback_features(fallback, config)\\n    features = _fit_feature_frames(features, config)\\n    return tf.expand_dims(tf.cast(features, tf.float32), axis=-1)\\n\\n\\ndef _mask_axis(\\n    feature_map: tf.Tensor,\\n    axis: int,\\n    mask_size: int,\\n) -> tf.Tensor:\\n    axis_length = tf.shape(feature_map)[axis]\\n    mask_size = tf.minimum(mask_size, axis_length)\\n    max_start = tf.maximum(axis_length - mask_size + 1, 1)\\n    start = tf.random.uniform([], 0, max_start, dtype=tf.int32)\\n\\n    if axis == 0:\\n        mask = tf.concat(\\n            [\\n                tf.ones([start, tf.shape(feature_map)[1], 1], dtype=feature_map.dtype),\\n                tf.zeros([mask_size, tf.shape(feature_map)[1], 1], dtype=feature_map.dtype),\\n                tf.ones(\\n                    [axis_length - start - mask_size, tf.shape(feature_map)[1], 1],\\n                    dtype=feature_map.dtype,\\n                ),\\n            ],\\n            axis=0,\\n        )\\n    else:\\n        mask = tf.concat(\\n            [\\n                tf.ones([tf.shape(feature_map)[0], start, 1], dtype=feature_map.dtype),\\n                tf.zeros([tf.shape(feature_map)[0], mask_size, 1], dtype=feature_map.dtype),\\n                tf.ones(\\n                    [tf.shape(feature_map)[0], axis_length - start - mask_size, 1],\\n                    dtype=feature_map.dtype,\\n                ),\\n            ],\\n            axis=1,\\n        )\\n    return feature_map * mask\\n\\n\\ndef apply_spec_augment(\\n    feature_map: tf.Tensor,\\n    specaugment_prob: float,\\n    time_mask_max: int,\\n    freq_mask_max: int,\\n) -> tf.Tensor:\\n    feature_map = tf.cast(feature_map, tf.float32)\\n    if specaugment_prob <= 0.0:\\n        return feature_map\\n\\n    def unchanged() -> tf.Tensor:\\n        return feature_map\\n\\n    def augmented() -> tf.Tensor:\\n        augmented_map = feature_map\\n        if time_mask_max > 0:\\n            time_size = tf.random.uniform([], 0, time_mask_max + 1, dtype=tf.int32)\\n            augmented_map = _mask_axis(augmented_map, axis=0, mask_size=time_size)\\n        if freq_mask_max > 0:\\n            freq_size = tf.random.uniform([], 0, freq_mask_max + 1, dtype=tf.int32)\\n            augmented_map = _mask_axis(augmented_map, axis=1, mask_size=freq_size)\\n        return augmented_map\\n\\n    should_augment = tf.less_equal(\\n        tf.random.uniform([], dtype=tf.float32),\\n        tf.cast(specaugment_prob, tf.float32),\\n    )\\n    return tf.cond(should_augment, augmented, unchanged)\\n", "code/training/kws_advanced/data.py": "from __future__ import annotations\\n\\nimport os\\nimport random\\nimport tarfile\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport tensorflow as tf\\n\\nfrom .config import ExperimentConfig\\nfrom .features import (\\n    apply_spec_augment,\\n    describe_frontend_runtime,\\n    extract_feature_map,\\n    pad_or_trim_waveform,\\n)\\n\\n\\nAUTOTUNE = tf.data.AUTOTUNE\\nRECORD_AUDIO = 0\\nRECORD_SILENCE = 1\\n\\n\\ndef _looks_like_speech_commands_root(path: Path) -> bool:\\n    required = [\\n        path / \\"validation_list.txt\\",\\n        path / \\"testing_list.txt\\",\\n    ]\\n    command_dirs = [\\n        path / \\"yes\\",\\n        path / \\"no\\",\\n    ]\\n    return all(item.exists() for item in required) and any(item.exists() for item in command_dirs)\\n\\n\\ndef _project_root(start: Path) -> Path:\\n    current = start.resolve()\\n    for candidate in [current, *current.parents]:\\n        if (candidate / \\"code\\").exists() and (candidate / \\"notes\\").exists():\\n            return candidate\\n    return start.resolve()\\n\\n\\ndef download_speech_commands(project_root: Path, experiment: ExperimentConfig) -> Path:\\n    env_dataset_root = os.environ.get(\\"SPEECH_COMMANDS_DATASET_ROOT\\", \\"\\").strip()\\n    env_archive_path = os.environ.get(\\"SPEECH_COMMANDS_ARCHIVE_PATH\\", \\"\\").strip()\\n    dataset_root_override = experiment.dataset.dataset_root_override.strip()\\n    archive_path_override = experiment.dataset.archive_path_override.strip()\\n\\n    explicit_dataset_roots = [\\n        Path(path)\\n        for path in [env_dataset_root, dataset_root_override]\\n        if path\\n    ]\\n    for candidate in explicit_dataset_roots:\\n        if candidate.exists():\\n            return candidate.resolve()\\n\\n    data_dir = project_root / experiment.dataset.data_dir\\n    data_dir.mkdir(parents=True, exist_ok=True)\\n\\n    extracted_dir = data_dir / \\"speech_commands_v0.02\\"\\n    if extracted_dir.exists():\\n        return extracted_dir.resolve()\\n    if _looks_like_speech_commands_root(data_dir):\\n        extracted_dir.mkdir(parents=True, exist_ok=True)\\n        for item in list(data_dir.iterdir()):\\n            if item == extracted_dir:\\n                continue\\n            item.rename(extracted_dir / item.name)\\n        return extracted_dir.resolve()\\n\\n    explicit_archives = [\\n        Path(path)\\n        for path in [env_archive_path, archive_path_override]\\n        if path\\n    ]\\n    for archive_path in explicit_archives:\\n        if archive_path.exists():\\n            extracted_dir.mkdir(parents=True, exist_ok=True)\\n            with tarfile.open(archive_path, \\"r:gz\\") as archive:\\n                archive.extractall(path=extracted_dir)\\n            if extracted_dir.exists():\\n                return extracted_dir.resolve()\\n\\n    try:\\n        archive_path = Path(\\n            tf.keras.utils.get_file(\\n                fname=\\"speech_commands_v0.02.tar.gz\\",\\n                origin=\\"https://download.tensorflow.org/data/speech_commands_v0.02.tar.gz\\",\\n                extract=True,\\n                cache_dir=str(data_dir),\\n                cache_subdir=\\"\\",\\n            )\\n        )\\n        extracted_candidate = archive_path.parent / \\"speech_commands_v0.02\\"\\n        if extracted_candidate.exists():\\n            return extracted_candidate.resolve()\\n    except Exception as exc:\\n        raise RuntimeError(\\n            \\"Could not prepare Speech Commands dataset. \\"\\n            \\"Use one of these options: \\"\\n            \\"1) set SPEECH_COMMANDS_DATASET_ROOT to an already extracted speech_commands_v0.02 folder; \\"\\n            \\"2) set SPEECH_COMMANDS_ARCHIVE_PATH to a local speech_commands_v0.02.tar.gz file; \\"\\n            \\"3) place speech_commands_v0.02 under <project>/data/ . \\"\\n            f\\"Original download error: {exc}\\"\\n        ) from exc\\n\\n    raise RuntimeError(\\n        \\"Speech Commands dataset was not found after download/extract. \\"\\n        \\"Set SPEECH_COMMANDS_DATASET_ROOT or SPEECH_COMMANDS_ARCHIVE_PATH explicitly.\\"\\n    )\\n\\n\\ndef _split_name_for(\\n    relative_path: str,\\n    validation_list: set[str],\\n    testing_list: set[str],\\n) -> str:\\n    if relative_path in validation_list:\\n        return \\"validation\\"\\n    if relative_path in testing_list:\\n        return \\"test\\"\\n    return \\"train\\"\\n\\n\\ndef _make_record(path: str, label_id: int, kind: int) -> dict[str, Any]:\\n    return {\\"path\\": path, \\"label_id\\": label_id, \\"kind\\": kind}\\n\\n\\ndef _maybe_take_fraction(\\n    records: list[dict[str, Any]],\\n    fraction: float,\\n    seed: int,\\n) -> list[dict[str, Any]]:\\n    if fraction >= 1.0:\\n        return list(records)\\n    fraction = max(fraction, 0.01)\\n    rng = random.Random(seed)\\n    copied = list(records)\\n    rng.shuffle(copied)\\n    keep = max(1, int(len(copied) * fraction))\\n    return copied[:keep]\\n\\n\\ndef build_record_index(\\n    project_root: Path,\\n    experiment: ExperimentConfig,\\n) -> dict[str, Any]:\\n    dataset_root = download_speech_commands(project_root, experiment)\\n    validation_list = set((dataset_root / \\"validation_list.txt\\").read_text().splitlines())\\n    testing_list = set((dataset_root / \\"testing_list.txt\\").read_text().splitlines())\\n\\n    per_split = {\\n        \\"train\\": {\\"targets\\": [], \\"unknown\\": []},\\n        \\"validation\\": {\\"targets\\": [], \\"unknown\\": []},\\n        \\"test\\": {\\"targets\\": [], \\"unknown\\": []},\\n    }\\n\\n    target_commands = set(experiment.commands)\\n    labels = experiment.label_to_index\\n    for wav_path in dataset_root.glob(\\"*/*.wav\\"):\\n        if wav_path.parent.name == \\"_background_noise_\\":\\n            continue\\n        relative = wav_path.relative_to(dataset_root).as_posix()\\n        split_name = _split_name_for(relative, validation_list, testing_list)\\n        word = wav_path.parent.name\\n        if word in target_commands:\\n            per_split[split_name][\\"targets\\"].append(\\n                _make_record(str(wav_path), labels[word], RECORD_AUDIO)\\n            )\\n        else:\\n            per_split[split_name][\\"unknown\\"].append(\\n                _make_record(str(wav_path), labels[\\"unknown\\"], RECORD_AUDIO)\\n            )\\n\\n    seed = experiment.dataset.seed\\n    merged_records: dict[str, list[dict[str, Any]]] = {}\\n    split_summary: dict[str, dict[str, int]] = {}\\n\\n    for offset, split_name in enumerate((\\"train\\", \\"validation\\", \\"test\\")):\\n        rng = random.Random(seed + offset)\\n        targets = list(per_split[split_name][\\"targets\\"])\\n        unknown = list(per_split[split_name][\\"unknown\\"])\\n        rng.shuffle(targets)\\n        rng.shuffle(unknown)\\n\\n        unknown_cap = int(len(targets) * experiment.dataset.unknown_ratio)\\n        mixed = targets + unknown[:unknown_cap]\\n        silence_count = int(len(mixed) * experiment.dataset.silence_ratio)\\n        mixed.extend(\\n            _make_record(\\"\\", labels[\\"silence\\"], RECORD_SILENCE) for _ in range(silence_count)\\n        )\\n        rng.shuffle(mixed)\\n\\n        fraction = (\\n            experiment.dataset.train_fraction\\n            if split_name == \\"train\\"\\n            else experiment.dataset.eval_fraction\\n        )\\n        mixed = _maybe_take_fraction(mixed, fraction=fraction, seed=seed + 100 + offset)\\n        merged_records[split_name] = mixed\\n        split_summary[split_name] = describe_records(mixed, experiment.all_labels)\\n\\n    return {\\n        \\"dataset_root\\": dataset_root,\\n        \\"records\\": merged_records,\\n        \\"summary\\": split_summary,\\n    }\\n\\n\\ndef describe_records(records: list[dict[str, Any]], labels: list[str]) -> dict[str, int]:\\n    counts = {label: 0 for label in labels}\\n    for record in records:\\n        label_id = int(record[\\"label_id\\"])\\n        counts[labels[label_id]] += 1\\n    return counts\\n\\n\\ndef _decode_wav(path: tf.Tensor) -> tf.Tensor:\\n    audio_binary = tf.io.read_file(path)\\n    waveform, _ = tf.audio.decode_wav(audio_binary, desired_channels=1)\\n    waveform = tf.squeeze(waveform, axis=-1)\\n    return waveform\\n\\n\\ndef _random_time_shift(waveform: tf.Tensor, shift_samples: int) -> tf.Tensor:\\n    if shift_samples <= 0:\\n        return waveform\\n    padded = tf.pad(waveform, [[shift_samples, shift_samples]])\\n    start = tf.random.uniform([], 0, 2 * shift_samples + 1, dtype=tf.int32)\\n    shifted = padded[start : start + tf.shape(waveform)[0]]\\n    shifted.set_shape(waveform.shape)\\n    return shifted\\n\\n\\ndef _augment_waveform(waveform: tf.Tensor, experiment: ExperimentConfig) -> tf.Tensor:\\n    waveform = tf.cast(waveform, tf.float32)\\n    shift_samples = (experiment.frontend.sample_rate * experiment.dataset.time_shift_ms) // 1000\\n    waveform = _random_time_shift(waveform, shift_samples=shift_samples)\\n    gain = tf.random.uniform(\\n        [],\\n        experiment.dataset.gain_min,\\n        experiment.dataset.gain_max,\\n        dtype=tf.float32,\\n    )\\n    waveform = waveform * gain\\n    if experiment.dataset.noise_stddev > 0.0:\\n        waveform = waveform + tf.random.normal(\\n            tf.shape(waveform),\\n            stddev=experiment.dataset.noise_stddev,\\n            dtype=tf.float32,\\n        )\\n    return tf.clip_by_value(waveform, -1.0, 1.0)\\n\\n\\ndef _prepare_example(\\n    sample: dict[str, tf.Tensor],\\n    experiment: ExperimentConfig,\\n    training: bool,\\n) -> tuple[dict[str, tf.Tensor], tf.Tensor]:\\n    kind = sample[\\"kind\\"]\\n\\n    def make_silence() -> tf.Tensor:\\n        return tf.zeros([experiment.frontend.desired_samples], dtype=tf.float32)\\n\\n    def decode_audio() -> tf.Tensor:\\n        return _decode_wav(sample[\\"path\\"])\\n\\n    waveform = tf.cond(\\n        tf.equal(kind, RECORD_SILENCE),\\n        make_silence,\\n        decode_audio,\\n    )\\n    waveform = pad_or_trim_waveform(waveform, experiment.student_frontend_config)\\n    if training:\\n        waveform = _augment_waveform(waveform, experiment)\\n\\n    student_feature_map = extract_feature_map(waveform, experiment.student_frontend_config)\\n    teacher_feature_map = extract_feature_map(waveform, experiment.teacher_frontend_config)\\n\\n    if training:\\n        student_feature_map = apply_spec_augment(\\n            student_feature_map,\\n            specaugment_prob=experiment.dataset.specaugment_prob,\\n            time_mask_max=experiment.dataset.time_mask_max,\\n            freq_mask_max=experiment.dataset.freq_mask_max,\\n        )\\n        teacher_feature_map = apply_spec_augment(\\n            teacher_feature_map,\\n            specaugment_prob=experiment.dataset.specaugment_prob,\\n            time_mask_max=experiment.dataset.time_mask_max,\\n            freq_mask_max=experiment.dataset.freq_mask_max,\\n        )\\n\\n    label = tf.one_hot(sample[\\"label_id\\"], depth=experiment.num_labels, dtype=tf.float32)\\n    return {\\n        \\"student\\": student_feature_map,\\n        \\"teacher\\": teacher_feature_map,\\n    }, label\\n\\n\\ndef _mixup_batch(\\n    features: dict[str, tf.Tensor],\\n    labels: tf.Tensor,\\n    alpha: float,\\n) -> tuple[dict[str, tf.Tensor], tf.Tensor]:\\n    if alpha <= 0.0:\\n        return features, labels\\n\\n    student_features = features[\\"student\\"]\\n    teacher_features = features[\\"teacher\\"]\\n    batch_size = tf.shape(student_features)[0]\\n    gamma_a = tf.random.gamma([batch_size], alpha=alpha, dtype=tf.float32)\\n    gamma_b = tf.random.gamma([batch_size], alpha=alpha, dtype=tf.float32)\\n    lam = gamma_a / (gamma_a + gamma_b)\\n    permutation = tf.random.shuffle(tf.range(batch_size))\\n\\n    mixed_student_features = (\\n        student_features * lam[:, tf.newaxis, tf.newaxis, tf.newaxis]\\n        + tf.gather(student_features, permutation)\\n        * (1.0 - lam)[:, tf.newaxis, tf.newaxis, tf.newaxis]\\n    )\\n    mixed_teacher_features = (\\n        teacher_features * lam[:, tf.newaxis, tf.newaxis, tf.newaxis]\\n        + tf.gather(teacher_features, permutation)\\n        * (1.0 - lam)[:, tf.newaxis, tf.newaxis, tf.newaxis]\\n    )\\n    mixed_labels = (\\n        labels * lam[:, tf.newaxis]\\n        + tf.gather(labels, permutation) * (1.0 - lam)[:, tf.newaxis]\\n    )\\n    return {\\n        \\"student\\": mixed_student_features,\\n        \\"teacher\\": mixed_teacher_features,\\n    }, mixed_labels\\n\\n\\ndef _records_to_tensor_slices(records: list[dict[str, Any]]) -> dict[str, list[Any]]:\\n    return {\\n        \\"path\\": [record[\\"path\\"] for record in records],\\n        \\"label_id\\": [int(record[\\"label_id\\"]) for record in records],\\n        \\"kind\\": [int(record[\\"kind\\"]) for record in records],\\n    }\\n\\n\\ndef make_dataset(\\n    records: list[dict[str, Any]],\\n    experiment: ExperimentConfig,\\n    training: bool,\\n) -> tf.data.Dataset:\\n    def map_example(sample: dict[str, tf.Tensor]) -> tuple[dict[str, tf.Tensor], tf.Tensor]:\\n        return _prepare_example(sample, experiment=experiment, training=training)\\n\\n    def map_mixup(features: dict[str, tf.Tensor], labels: tf.Tensor) -> tuple[dict[str, tf.Tensor], tf.Tensor]:\\n        return _mixup_batch(\\n            features,\\n            labels,\\n            alpha=experiment.dataset.mixup_alpha,\\n        )\\n\\n    slices = _records_to_tensor_slices(records)\\n    dataset = tf.data.Dataset.from_tensor_slices(slices)\\n    if training:\\n        dataset = dataset.shuffle(\\n            buffer_size=min(len(records), experiment.dataset.shuffle_buffer),\\n            seed=experiment.dataset.seed,\\n            reshuffle_each_iteration=True,\\n        )\\n    dataset = dataset.map(map_example, num_parallel_calls=AUTOTUNE)\\n    if not training and experiment.dataset.cache_validation:\\n        dataset = dataset.cache()\\n    dataset = dataset.batch(experiment.dataset.batch_size)\\n    if training and experiment.dataset.mixup_alpha > 0.0:\\n        dataset = dataset.map(map_mixup, num_parallel_calls=AUTOTUNE)\\n    return dataset.prefetch(AUTOTUNE)\\n\\n\\ndef _select_feature_view(\\n    dataset: tf.data.Dataset,\\n    key: str,\\n) -> tf.data.Dataset:\\n    def select_view(features: dict[str, tf.Tensor], labels: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:\\n        return features[key], labels\\n\\n    return dataset.map(select_view, num_parallel_calls=AUTOTUNE)\\n\\n\\ndef prepare_datasets(\\n    project_root: Path | None,\\n    experiment: ExperimentConfig,\\n) -> dict[str, Any]:\\n    if project_root is None:\\n        project_root = _project_root(Path.cwd())\\n    index = build_record_index(project_root, experiment)\\n    dual_datasets = {\\n        split_name: make_dataset(records, experiment=experiment, training=(split_name == \\"train\\"))\\n        for split_name, records in index[\\"records\\"].items()\\n    }\\n    datasets: dict[str, tf.data.Dataset] = {}\\n    for split_name, dataset in dual_datasets.items():\\n        datasets[f\\"distillation_{split_name}\\"] = dataset\\n        datasets[f\\"teacher_{split_name}\\"] = _select_feature_view(dataset, \\"teacher\\")\\n        datasets[f\\"student_{split_name}\\"] = _select_feature_view(dataset, \\"student\\")\\n    train_size = len(index[\\"records\\"][\\"train\\"])\\n    steps_per_epoch = max(1, train_size // experiment.dataset.batch_size)\\n    return {\\n        \\"project_root\\": project_root,\\n        \\"dataset_root\\": index[\\"dataset_root\\"],\\n        \\"records\\": index[\\"records\\"],\\n        \\"summary\\": index[\\"summary\\"],\\n        \\"frontend\\": {\\n            \\"student\\": describe_frontend_runtime(experiment.student_frontend_config),\\n            \\"teacher\\": describe_frontend_runtime(experiment.teacher_frontend_config),\\n            \\"teacher_student_frontend_match\\": experiment.teacher_frontend_config == experiment.student_frontend_config,\\n        },\\n        \\"datasets\\": datasets,\\n        \\"steps_per_epoch\\": steps_per_epoch,\\n    }\\n", "code/training/kws_advanced/models.py": "from __future__ import annotations\\n\\nfrom typing import Iterable\\n\\nimport tensorflow as tf\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef _round_filters(filters: int, width: float) -> int:\\n    scaled = int(round(filters * width))\\n    return max(8, int(round(scaled / 8.0) * 8))\\n\\n\\ndef _conv_bn_relu(\\n    x: tf.Tensor,\\n    filters: int,\\n    kernel_size: tuple[int, int],\\n    strides: tuple[int, int],\\n    name: str,\\n) -> tf.Tensor:\\n    x = tf.keras.layers.Conv2D(\\n        filters,\\n        kernel_size=kernel_size,\\n        strides=strides,\\n        padding=\\"same\\",\\n        use_bias=False,\\n        name=f\\"{name}_conv\\",\\n    )(x)\\n    x = tf.keras.layers.BatchNormalization(name=f\\"{name}_bn\\")(x)\\n    return tf.keras.layers.ReLU(max_value=6.0, name=f\\"{name}_relu\\")(x)\\n\\n\\n@tf.keras.utils.register_keras_serializable(package=\\"kws_advanced\\")\\nclass LearnableClassToken(tf.keras.layers.Layer):\\n    def build(self, input_shape) -> None:\\n        embedding_dim = input_shape[-1]\\n        if embedding_dim is None:\\n            raise ValueError(\\"LearnableClassToken requires a known embedding dimension.\\")\\n        self.class_token = self.add_weight(\\n            name=\\"class_token\\",\\n            shape=(1, 1, embedding_dim),\\n            initializer=tf.keras.initializers.TruncatedNormal(stddev=0.02),\\n            trainable=True,\\n        )\\n        super().build(input_shape)\\n\\n    def call(self, inputs):\\n        batch_size = tf.shape(inputs)[0]\\n        class_token = tf.broadcast_to(self.class_token, [batch_size, 1, tf.shape(inputs)[-1]])\\n        return tf.concat([class_token, inputs], axis=1)\\n\\n    def get_config(self) -> dict:\\n        return super().get_config()\\n\\n\\n@tf.keras.utils.register_keras_serializable(package=\\"kws_advanced\\")\\nclass LearnablePositionEmbedding(tf.keras.layers.Layer):\\n    def build(self, input_shape) -> None:\\n        sequence_length = input_shape[1]\\n        embedding_dim = input_shape[2]\\n        if sequence_length is None or embedding_dim is None:\\n            raise ValueError(\\"LearnablePositionEmbedding requires known sequence and embedding dimensions.\\")\\n        self.position_embedding = self.add_weight(\\n            name=\\"position_embedding\\",\\n            shape=(1, sequence_length, embedding_dim),\\n            initializer=tf.keras.initializers.TruncatedNormal(stddev=0.02),\\n            trainable=True,\\n        )\\n        super().build(input_shape)\\n\\n    def call(self, inputs):\\n        return inputs + self.position_embedding\\n\\n    def get_config(self) -> dict:\\n        return super().get_config()\\n\\n\\n@tf.keras.utils.register_keras_serializable(package=\\"kws_advanced\\")\\nclass TakeClassToken(tf.keras.layers.Layer):\\n    def call(self, inputs):\\n        return inputs[:, 0]\\n\\n    def get_config(self) -> dict:\\n        return super().get_config()\\n\\n\\n@tf.keras.utils.register_keras_serializable(package=\\"kws_advanced\\")\\nclass SubSpectralNormalization(tf.keras.layers.Layer):\\n    def __init__(\\n        self,\\n        spec_groups: int = 5,\\n        momentum: float = 0.99,\\n        epsilon: float = 1e-3,\\n        **kwargs,\\n    ) -> None:\\n        super().__init__(**kwargs)\\n        self.spec_groups = spec_groups\\n        self.momentum = momentum\\n        self.epsilon = epsilon\\n        self._bn: tf.keras.layers.BatchNormalization | None = None\\n\\n    def build(self, input_shape) -> None:\\n        channels = input_shape[-1]\\n        if channels is None:\\n            raise ValueError(\\"SubSpectralNormalization requires a known channel dimension.\\")\\n        self._bn = tf.keras.layers.BatchNormalization(\\n            axis=-1,\\n            momentum=self.momentum,\\n            epsilon=self.epsilon,\\n            name=f\\"{self.name}_bn\\",\\n        )\\n        self._bn.build((input_shape[0], None, input_shape[2], channels * self.spec_groups))\\n        super().build(input_shape)\\n\\n    def call(self, inputs, training=None):\\n        if self._bn is None:\\n            raise RuntimeError(\\"SubSpectralNormalization was not built.\\")\\n        shape = tf.shape(inputs)\\n        batch_size = shape[0]\\n        freq_bins = shape[1]\\n        time_bins = shape[2]\\n        channels = shape[3]\\n        tf.debugging.assert_equal(\\n            tf.math.floormod(freq_bins, self.spec_groups),\\n            0,\\n            message=\\"Frequency bins must be divisible by spec_groups.\\",\\n        )\\n        reshaped = tf.reshape(\\n            inputs,\\n            [batch_size, freq_bins // self.spec_groups, time_bins, channels * self.spec_groups],\\n        )\\n        normalized = self._bn(reshaped, training=training)\\n        return tf.reshape(normalized, [batch_size, freq_bins, time_bins, channels])\\n\\n    def get_config(self) -> dict:\\n        config = super().get_config()\\n        config.update(\\n            {\\n                \\"spec_groups\\": self.spec_groups,\\n                \\"momentum\\": self.momentum,\\n                \\"epsilon\\": self.epsilon,\\n            }\\n        )\\n        return config\\n\\n\\ndef _factorized_residual_block(\\n    x: tf.Tensor,\\n    filters: int,\\n    temporal_kernel: int,\\n    freq_kernel: int,\\n    strides: tuple[int, int],\\n    residual: bool,\\n    name: str,\\n) -> tf.Tensor:\\n    shortcut = x\\n    x = tf.keras.layers.DepthwiseConv2D(\\n        kernel_size=(temporal_kernel, 1),\\n        strides=strides,\\n        padding=\\"same\\",\\n        use_bias=False,\\n        name=f\\"{name}_dw_time\\",\\n    )(x)\\n    x = tf.keras.layers.BatchNormalization(name=f\\"{name}_dw_time_bn\\")(x)\\n    x = tf.keras.layers.ReLU(max_value=6.0, name=f\\"{name}_dw_time_relu\\")(x)\\n\\n    x = tf.keras.layers.DepthwiseConv2D(\\n        kernel_size=(1, freq_kernel),\\n        strides=(1, 1),\\n        padding=\\"same\\",\\n        use_bias=False,\\n        name=f\\"{name}_dw_freq\\",\\n    )(x)\\n    x = tf.keras.layers.BatchNormalization(name=f\\"{name}_dw_freq_bn\\")(x)\\n    x = tf.keras.layers.ReLU(max_value=6.0, name=f\\"{name}_dw_freq_relu\\")(x)\\n\\n    x = tf.keras.layers.Conv2D(\\n        filters,\\n        kernel_size=(1, 1),\\n        strides=(1, 1),\\n        padding=\\"same\\",\\n        use_bias=False,\\n        name=f\\"{name}_pw\\",\\n    )(x)\\n    x = tf.keras.layers.BatchNormalization(name=f\\"{name}_pw_bn\\")(x)\\n\\n    if residual:\\n        if shortcut.shape[-1] != filters or strides != (1, 1):\\n            shortcut = tf.keras.layers.Conv2D(\\n                filters,\\n                kernel_size=(1, 1),\\n                strides=strides,\\n                padding=\\"same\\",\\n                use_bias=False,\\n                name=f\\"{name}_skip_conv\\",\\n            )(shortcut)\\n            shortcut = tf.keras.layers.BatchNormalization(name=f\\"{name}_skip_bn\\")(shortcut)\\n        x = tf.keras.layers.Add(name=f\\"{name}_add\\")([x, shortcut])\\n\\n    return tf.keras.layers.ReLU(max_value=6.0, name=f\\"{name}_out\\")(x)\\n\\n\\ndef _bcresnet_head(x: tf.Tensor, base_channels: int, name: str) -> tf.Tensor:\\n    x = tf.keras.layers.Permute((2, 1, 3), name=f\\"{name}_permute_to_freq_time\\")(x)\\n    x = tf.keras.layers.Conv2D(\\n        base_channels * 2,\\n        kernel_size=(5, 5),\\n        strides=(2, 1),\\n        padding=\\"same\\",\\n        use_bias=False,\\n        name=f\\"{name}_head_conv\\",\\n    )(x)\\n    x = tf.keras.layers.BatchNormalization(name=f\\"{name}_head_bn\\")(x)\\n    return tf.keras.layers.ReLU(name=f\\"{name}_head_relu\\")(x)\\n\\n\\ndef _bcresnet_block(\\n    x: tf.Tensor,\\n    out_channels: int,\\n    stage_index: int,\\n    use_stride: bool,\\n    name: str,\\n) -> tf.Tensor:\\n    shortcut = x\\n    in_channels = int(x.shape[-1])\\n    transition_block = in_channels != out_channels\\n\\n    if transition_block:\\n        x = tf.keras.layers.Conv2D(\\n            out_channels,\\n            kernel_size=(1, 1),\\n            strides=(1, 1),\\n            padding=\\"same\\",\\n            use_bias=False,\\n            name=f\\"{name}_transition_conv\\",\\n        )(x)\\n        x = tf.keras.layers.BatchNormalization(name=f\\"{name}_transition_bn\\")(x)\\n        x = tf.keras.layers.ReLU(name=f\\"{name}_transition_relu\\")(x)\\n\\n    stride = (2, 1) if use_stride else (1, 1)\\n    x = tf.keras.layers.DepthwiseConv2D(\\n        kernel_size=(3, 1),\\n        strides=stride,\\n        padding=\\"same\\",\\n        use_bias=False,\\n        name=f\\"{name}_f2_depthwise\\",\\n    )(x)\\n    x = SubSpectralNormalization(spec_groups=5, name=f\\"{name}_ssn\\")(x)\\n    aux_2d_res = x\\n\\n    freq_bins = int(x.shape[1])\\n    x = tf.keras.layers.AveragePooling2D(\\n        pool_size=(freq_bins, 1),\\n        name=f\\"{name}_freq_pool\\",\\n    )(x)\\n    x = tf.keras.layers.DepthwiseConv2D(\\n        kernel_size=(1, 3),\\n        strides=(1, 1),\\n        padding=\\"same\\",\\n        dilation_rate=(1, 2 ** stage_index),\\n        use_bias=False,\\n        name=f\\"{name}_f1_depthwise\\",\\n    )(x)\\n    x = tf.keras.layers.BatchNormalization(name=f\\"{name}_f1_bn\\")(x)\\n    x = tf.keras.layers.Activation(\\"swish\\", name=f\\"{name}_f1_swish\\")(x)\\n    x = tf.keras.layers.Conv2D(\\n        out_channels,\\n        kernel_size=(1, 1),\\n        strides=(1, 1),\\n        padding=\\"same\\",\\n        use_bias=False,\\n        name=f\\"{name}_f1_pointwise\\",\\n    )(x)\\n    x = tf.keras.layers.SpatialDropout2D(0.1, name=f\\"{name}_dropout\\")(x)\\n    x = tf.keras.layers.Add(name=f\\"{name}_broadcast_add\\")([x, aux_2d_res])\\n\\n    if not transition_block:\\n        x = tf.keras.layers.Add(name=f\\"{name}_residual_add\\")([x, shortcut])\\n\\n    return tf.keras.layers.ReLU(name=f\\"{name}_out\\")(x)\\n\\n\\ndef _kwt_encoder_block(\\n    x: tf.Tensor,\\n    embedding_dim: int,\\n    num_heads: int,\\n    mlp_dim: int,\\n    dropout: float,\\n    name: str,\\n) -> tf.Tensor:\\n    attention = tf.keras.layers.MultiHeadAttention(\\n        num_heads=num_heads,\\n        key_dim=embedding_dim // num_heads,\\n        dropout=dropout,\\n        name=f\\"{name}_attention\\",\\n    )(x, x)\\n    x = tf.keras.layers.Add(name=f\\"{name}_attention_add\\")([x, attention])\\n    x = tf.keras.layers.LayerNormalization(epsilon=1e-6, name=f\\"{name}_attention_norm\\")(x)\\n\\n    mlp = tf.keras.layers.Dense(mlp_dim, activation=tf.nn.gelu, name=f\\"{name}_mlp_expand\\")(x)\\n    mlp = tf.keras.layers.Dropout(dropout, name=f\\"{name}_mlp_dropout1\\")(mlp)\\n    mlp = tf.keras.layers.Dense(embedding_dim, name=f\\"{name}_mlp_project\\")(mlp)\\n    mlp = tf.keras.layers.Dropout(dropout, name=f\\"{name}_mlp_dropout2\\")(mlp)\\n\\n    x = tf.keras.layers.Add(name=f\\"{name}_mlp_add\\")([x, mlp])\\n    x = tf.keras.layers.LayerNormalization(epsilon=1e-6, name=f\\"{name}_out\\")(x)\\n    return x\\n\\n\\ndef build_bcresnet_teacher(\\n    input_shape: tuple[int, int, int],\\n    num_classes: int,\\n    width: float,\\n    dropout: float,\\n) -> tf.keras.Model:\\n    del dropout\\n    base_channels = max(8, int(round(16 * width)))\\n    channels = [\\n        base_channels * 2,\\n        base_channels,\\n        int(round(base_channels * 1.5)),\\n        base_channels * 2,\\n        int(round(base_channels * 2.5)),\\n        base_channels * 4,\\n    ]\\n    repeats = (2, 2, 4, 4)\\n    stride_stages = {1, 2}\\n\\n    inputs = tf.keras.Input(shape=input_shape, name=\\"input_features\\")\\n    x = _bcresnet_head(inputs, base_channels=base_channels, name=\\"bcresnet\\")\\n\\n    for stage_index, repeat_count in enumerate(repeats):\\n        out_channels = channels[stage_index + 1]\\n        for block_index in range(repeat_count):\\n            x = _bcresnet_block(\\n                x,\\n                out_channels=out_channels,\\n                stage_index=stage_index,\\n                use_stride=(stage_index in stride_stages and block_index == 0),\\n                name=f\\"bcresnet_stage{stage_index + 1}_block{block_index + 1}\\",\\n            )\\n\\n    x = tf.keras.layers.ZeroPadding2D(padding=((0, 0), (2, 2)), name=\\"bcresnet_classifier_pad\\")(x)\\n    x = tf.keras.layers.DepthwiseConv2D(\\n        kernel_size=(5, 5),\\n        strides=(1, 1),\\n        padding=\\"valid\\",\\n        use_bias=False,\\n        name=\\"bcresnet_classifier_depthwise\\",\\n    )(x)\\n    x = tf.keras.layers.Conv2D(\\n        channels[-1],\\n        kernel_size=(1, 1),\\n        strides=(1, 1),\\n        padding=\\"same\\",\\n        use_bias=False,\\n        name=\\"bcresnet_classifier_pointwise\\",\\n    )(x)\\n    x = tf.keras.layers.BatchNormalization(name=\\"bcresnet_classifier_bn\\")(x)\\n    x = tf.keras.layers.ReLU(name=\\"bcresnet_classifier_relu\\")(x)\\n    x = tf.keras.layers.GlobalAveragePooling2D(name=\\"bcresnet_global_pool\\")(x)\\n    outputs = tf.keras.layers.Dense(num_classes, activation=None, name=\\"bcresnet_logits\\")(x)\\n    return tf.keras.Model(inputs=inputs, outputs=outputs, name=\\"teacher_bcresnet\\")\\n\\n\\ndef build_kwt_teacher(\\n    input_shape: tuple[int, int, int],\\n    num_classes: int,\\n    width: float,\\n    dropout: float,\\n) -> tf.keras.Model:\\n    if input_shape[-1] != 1:\\n        raise ValueError(\\"KWT teacher expects a single-channel feature map.\\")\\n\\n    embedding_dim = max(64, int(round(192 * width / 64.0) * 64))\\n    num_heads = max(1, embedding_dim // 64)\\n    mlp_dim = embedding_dim * 4\\n    num_layers = 12\\n\\n    inputs = tf.keras.Input(shape=input_shape, name=\\"input_features\\")\\n    x = tf.keras.layers.Reshape((input_shape[0], input_shape[1]), name=\\"kwt_frame_tokens\\")(inputs)\\n    x = tf.keras.layers.Dense(embedding_dim, use_bias=True, name=\\"kwt_patch_projection\\")(x)\\n    x = LearnableClassToken(name=\\"kwt_class_token\\")(x)\\n    x = LearnablePositionEmbedding(name=\\"kwt_position_embedding\\")(x)\\n    x = tf.keras.layers.Dropout(dropout, name=\\"kwt_input_dropout\\")(x)\\n\\n    for block_index in range(num_layers):\\n        x = _kwt_encoder_block(\\n            x,\\n            embedding_dim=embedding_dim,\\n            num_heads=num_heads,\\n            mlp_dim=mlp_dim,\\n            dropout=dropout,\\n            name=f\\"kwt_block{block_index + 1}\\",\\n        )\\n\\n    cls_token = TakeClassToken(name=\\"kwt_cls_slice\\")(x)\\n    cls_token = tf.keras.layers.LayerNormalization(epsilon=1e-6, name=\\"kwt_cls_norm\\")(cls_token)\\n    outputs = tf.keras.layers.Dense(num_classes, activation=None, name=\\"kwt_logits\\")(cls_token)\\n    return tf.keras.Model(inputs=inputs, outputs=outputs, name=\\"teacher_kwt\\")\\n\\n\\ndef _pool_head(\\n    x: tf.Tensor,\\n    dropout: float,\\n    num_classes: int,\\n    name: str,\\n) -> tf.Tensor:\\n    head_filters = max(int(x.shape[-1]), 32)\\n    x = _conv_bn_relu(x, head_filters, kernel_size=(1, 1), strides=(1, 1), name=f\\"{name}_head\\")\\n    pool_height = int(x.shape[1])\\n    pool_width = int(x.shape[2])\\n    x = tf.keras.layers.AveragePooling2D(\\n        pool_size=(pool_height, pool_width),\\n        name=f\\"{name}_avgpool\\",\\n    )(x)\\n    x = tf.keras.layers.Flatten(name=f\\"{name}_flatten\\")(x)\\n    x = tf.keras.layers.Dropout(dropout, name=f\\"{name}_dropout\\")(x)\\n    return tf.keras.layers.Dense(num_classes, activation=None, name=f\\"{name}_logits\\")(x)\\n\\n\\ndef build_baseline_cnn(\\n    input_shape: tuple[int, int, int],\\n    num_classes: int,\\n    width: float,\\n    dropout: float,\\n) -> tf.keras.Model:\\n    inputs = tf.keras.Input(shape=input_shape, name=\\"input_features\\")\\n    x = _conv_bn_relu(inputs, _round_filters(24, width), (3, 3), (2, 2), name=\\"stem\\")\\n    x = _conv_bn_relu(x, _round_filters(32, width), (3, 3), (2, 1), name=\\"block1\\")\\n    x = _conv_bn_relu(x, _round_filters(48, width), (3, 3), (2, 2), name=\\"block2\\")\\n    outputs = _pool_head(x, dropout=dropout, num_classes=num_classes, name=\\"baseline\\")\\n    return tf.keras.Model(inputs=inputs, outputs=outputs, name=\\"baseline_cnn\\")\\n\\n\\ndef build_factorized_dscnn(\\n    input_shape: tuple[int, int, int],\\n    num_classes: int,\\n    width: float,\\n    dropout: float,\\n    block_filters: Iterable[int],\\n    block_repeats: Iterable[int],\\n    name: str,\\n) -> tf.keras.Model:\\n    inputs = tf.keras.Input(shape=input_shape, name=\\"input_features\\")\\n    x = _conv_bn_relu(\\n        inputs,\\n        _round_filters(24, width),\\n        kernel_size=(5, 3),\\n        strides=(2, 2),\\n        name=\\"stem\\",\\n    )\\n\\n    for stage_index, (filters, repeats) in enumerate(zip(block_filters, block_repeats)):\\n        rounded_filters = _round_filters(filters, width)\\n        for block_index in range(repeats):\\n            strides = (1, 1)\\n            if stage_index > 0 and block_index == 0:\\n                strides = (2, 1) if stage_index == 1 else (2, 2)\\n            x = _factorized_residual_block(\\n                x,\\n                filters=rounded_filters,\\n                temporal_kernel=5 if stage_index < 2 else 3,\\n                freq_kernel=3,\\n                strides=strides,\\n                residual=True,\\n                name=f\\"stage{stage_index + 1}_block{block_index + 1}\\",\\n            )\\n\\n    outputs = _pool_head(x, dropout=dropout, num_classes=num_classes, name=name)\\n    return tf.keras.Model(inputs=inputs, outputs=outputs, name=name)\\n\\n\\ndef build_model_by_name(\\n    model_name: str,\\n    input_shape: tuple[int, int, int],\\n    num_classes: int,\\n    width: float,\\n    dropout: float,\\n) -> tf.keras.Model:\\n    if model_name == \\"baseline_cnn\\":\\n        return build_baseline_cnn(input_shape, num_classes, width=width, dropout=dropout)\\n    if model_name == \\"teacher_factorized_dscnn\\":\\n        return build_factorized_dscnn(\\n            input_shape,\\n            num_classes,\\n            width=width,\\n            dropout=dropout,\\n            block_filters=(32, 48, 64),\\n            block_repeats=(2, 2, 2),\\n            name=\\"teacher_factorized_dscnn\\",\\n        )\\n    if model_name == \\"teacher_factorized_dscnn_xl\\":\\n        return build_factorized_dscnn(\\n            input_shape,\\n            num_classes,\\n            width=width,\\n            dropout=dropout,\\n            block_filters=(48, 72, 96),\\n            block_repeats=(2, 3, 2),\\n            name=\\"teacher_factorized_dscnn_xl\\",\\n        )\\n    if model_name == \\"teacher_bcresnet\\":\\n        return build_bcresnet_teacher(\\n            input_shape,\\n            num_classes,\\n            width=width,\\n            dropout=dropout,\\n        )\\n    if model_name == \\"teacher_kwt\\":\\n        return build_kwt_teacher(\\n            input_shape,\\n            num_classes,\\n            width=width,\\n            dropout=dropout,\\n        )\\n    if model_name == \\"student_factorized_dscnn\\":\\n        return build_factorized_dscnn(\\n            input_shape,\\n            num_classes,\\n            width=width,\\n            dropout=dropout,\\n            block_filters=(24, 32, 48),\\n            block_repeats=(1, 2, 1),\\n            name=\\"student_factorized_dscnn\\",\\n        )\\n    if model_name == \\"student_factorized_dscnn_v2\\":\\n        return build_factorized_dscnn(\\n            input_shape,\\n            num_classes,\\n            width=width,\\n            dropout=dropout,\\n            block_filters=(32, 48, 64),\\n            block_repeats=(2, 2, 1),\\n            name=\\"student_factorized_dscnn_v2\\",\\n        )\\n    raise KeyError(f\\"Unknown model name: {model_name}\\")\\n\\n\\ndef build_teacher_model(experiment: ExperimentConfig) -> tf.keras.Model:\\n    return build_model_by_name(\\n        model_name=experiment.model.teacher_name,\\n        input_shape=experiment.teacher_feature_shape,\\n        num_classes=experiment.num_labels,\\n        width=experiment.model.teacher_width,\\n        dropout=experiment.model.teacher_dropout,\\n    )\\n\\n\\ndef build_student_model(experiment: ExperimentConfig) -> tf.keras.Model:\\n    return build_model_by_name(\\n        model_name=experiment.model.student_name,\\n        input_shape=experiment.student_feature_shape,\\n        num_classes=experiment.num_labels,\\n        width=experiment.model.student_width,\\n        dropout=experiment.model.student_dropout,\\n    )\\n\\n\\ndef _shape_list(value) -> list[int | None]:\\n    if value is None:\\n        return []\\n    if isinstance(value, (list, tuple)) and value and not hasattr(value, \\"shape\\"):\\n        value = value[0]\\n    if hasattr(value, \\"shape\\"):\\n        shape = value.shape\\n    else:\\n        shape = value\\n    try:\\n        return list(tf.TensorShape(shape).as_list())\\n    except Exception:\\n        return []\\n\\n\\ndef estimate_maccs(model: tf.keras.Model) -> int:\\n    total = 0\\n    for layer in model.layers:\\n        if isinstance(layer, tf.keras.layers.Conv2D):\\n            output_shape = _shape_list(layer.output)\\n            input_shape = _shape_list(layer.input)\\n            if len(output_shape) != 4 or len(input_shape) != 4:\\n                continue\\n            _, out_h, out_w, out_c = output_shape\\n            kernel_h, kernel_w = layer.kernel_size\\n            in_c = input_shape[-1]\\n            if None in (out_h, out_w, out_c, in_c):\\n                continue\\n            total += int(out_h) * int(out_w) * kernel_h * kernel_w * in_c * int(out_c)\\n        elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):\\n            output_shape = _shape_list(layer.output)\\n            input_shape = _shape_list(layer.input)\\n            if len(output_shape) != 4 or len(input_shape) != 4:\\n                continue\\n            _, out_h, out_w, out_c = output_shape\\n            kernel_h, kernel_w = layer.kernel_size\\n            in_c = input_shape[-1]\\n            depth_multiplier = int(layer.depth_multiplier)\\n            if None in (out_h, out_w, out_c, in_c):\\n                continue\\n            total += int(out_h) * int(out_w) * kernel_h * kernel_w * in_c * depth_multiplier\\n        elif isinstance(layer, tf.keras.layers.Dense):\\n            input_shape = _shape_list(layer.input)\\n            if not input_shape or input_shape[-1] is None:\\n                continue\\n            total += int(input_shape[-1]) * int(layer.units)\\n        elif isinstance(layer, tf.keras.layers.MultiHeadAttention):\\n            if not isinstance(layer.input, (list, tuple)) or len(layer.input) < 2:\\n                continue\\n            query_shape = _shape_list(layer.input[0])\\n            value_shape = _shape_list(layer.input[1])\\n            output_shape = _shape_list(layer.output)\\n            if len(query_shape) != 3 or len(value_shape) != 3 or len(output_shape) != 3:\\n                continue\\n            _, query_length, query_dim = query_shape\\n            _, value_length, value_dim = value_shape\\n            _, _, output_dim = output_shape\\n            key_dim = int(layer.key_dim)\\n            value_head_dim = int(layer.value_dim or layer.key_dim)\\n            num_heads = int(layer.num_heads)\\n            if None in (query_length, query_dim, value_length, value_dim, output_dim):\\n                continue\\n            total += int(query_length) * int(query_dim) * num_heads * key_dim\\n            total += int(value_length) * int(value_dim) * num_heads * key_dim\\n            total += int(value_length) * int(value_dim) * num_heads * value_head_dim\\n            total += int(query_length) * num_heads * value_head_dim * int(output_dim)\\n            total += num_heads * int(query_length) * int(value_length) * (key_dim + value_head_dim)\\n    return total\\n\\n\\ndef required_tflite_ops(model_name: str) -> list[str]:\\n    if model_name not in {\\n        \\"baseline_cnn\\",\\n        \\"teacher_factorized_dscnn\\",\\n        \\"teacher_factorized_dscnn_xl\\",\\n        \\"teacher_bcresnet\\",\\n        \\"teacher_kwt\\",\\n        \\"student_factorized_dscnn\\",\\n        \\"student_factorized_dscnn_v2\\",\\n    }:\\n        raise KeyError(f\\"Unknown model name: {model_name}\\")\\n    if model_name == \\"teacher_kwt\\":\\n        return [\\n            \\"FULLY_CONNECTED\\",\\n            \\"BATCH_MATMUL\\",\\n            \\"SOFTMAX\\",\\n            \\"RESHAPE\\",\\n            \\"ADD\\",\\n            \\"MUL\\",\\n            \\"TRANSPOSE\\",\\n        ]\\n    return [\\n        \\"CONV_2D\\",\\n        \\"DEPTHWISE_CONV_2D\\",\\n        \\"ADD\\",\\n        \\"AVERAGE_POOL_2D\\",\\n        \\"RESHAPE\\",\\n        \\"FULLY_CONNECTED\\",\\n    ]\\n", "code/training/kws_advanced/distillation.py": "from __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport tensorflow as tf\\n\\nfrom .config import ExperimentConfig\\n\\n\\n@tf.keras.utils.register_keras_serializable(package=\\"kws_advanced\\")\\nclass WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):\\n    def __init__(\\n        self,\\n        initial_learning_rate: float,\\n        decay_steps: int,\\n        warmup_steps: int = 0,\\n        alpha: float = 0.1,\\n    ) -> None:\\n        super().__init__()\\n        self.initial_learning_rate = float(initial_learning_rate)\\n        self.decay_steps = int(max(1, decay_steps))\\n        self.warmup_steps = int(max(0, warmup_steps))\\n        self.alpha = float(alpha)\\n\\n    def __call__(self, step):\\n        step = tf.cast(step, tf.float32)\\n        initial_lr = tf.cast(self.initial_learning_rate, tf.float32)\\n        warmup_steps = tf.cast(max(1, self.warmup_steps), tf.float32)\\n\\n        def warmup() -> tf.Tensor:\\n            return initial_lr * (step + 1.0) / warmup_steps\\n\\n        def cosine() -> tf.Tensor:\\n            adjusted_step = tf.maximum(0.0, step - float(self.warmup_steps))\\n            adjusted_decay_steps = float(max(1, self.decay_steps - self.warmup_steps))\\n            cosine_decay = 0.5 * (\\n                1.0\\n                + tf.cos(\\n                    tf.constant(3.141592653589793, dtype=tf.float32)\\n                    * adjusted_step\\n                    / adjusted_decay_steps\\n                )\\n            )\\n            decayed = (1.0 - self.alpha) * cosine_decay + self.alpha\\n            return initial_lr * decayed\\n\\n        if self.warmup_steps <= 0:\\n            return cosine()\\n        return tf.cond(step < float(self.warmup_steps), warmup, cosine)\\n\\n    def get_config(self) -> dict:\\n        return {\\n            \\"initial_learning_rate\\": self.initial_learning_rate,\\n            \\"decay_steps\\": self.decay_steps,\\n            \\"warmup_steps\\": self.warmup_steps,\\n            \\"alpha\\": self.alpha,\\n        }\\n\\n\\ndef _build_optimizer(\\n    experiment: ExperimentConfig,\\n    learning_rate: float,\\n    total_steps: int,\\n    warmup_steps: int = 0,\\n) -> tf.keras.optimizers.Optimizer:\\n    total_steps = max(1, total_steps)\\n    schedule = WarmupCosineDecay(\\n        initial_learning_rate=learning_rate,\\n        decay_steps=total_steps,\\n        warmup_steps=warmup_steps,\\n        alpha=0.1,\\n    )\\n    optimizer_name = experiment.train.optimizer_name.lower()\\n    common_kwargs = {\\n        \\"learning_rate\\": schedule,\\n        \\"clipnorm\\": experiment.train.clipnorm,\\n    }\\n    if optimizer_name == \\"adamw\\":\\n        return tf.keras.optimizers.AdamW(\\n            weight_decay=experiment.train.weight_decay,\\n            **common_kwargs,\\n        )\\n    if optimizer_name == \\"adam\\":\\n        return tf.keras.optimizers.Adam(**common_kwargs)\\n    raise KeyError(f\\"Unsupported optimizer_name: {experiment.train.optimizer_name}\\")\\n\\n\\ndef compile_classifier(\\n    model: tf.keras.Model,\\n    experiment: ExperimentConfig,\\n    learning_rate: float,\\n    epochs: int,\\n    steps_per_epoch: int,\\n    label_smoothing: float | None = None,\\n    warmup_epochs: int = 0,\\n) -> tf.keras.Model:\\n    optimizer = _build_optimizer(\\n        experiment=experiment,\\n        learning_rate=learning_rate,\\n        total_steps=epochs * steps_per_epoch,\\n        warmup_steps=warmup_epochs * steps_per_epoch,\\n    )\\n    model.compile(\\n        optimizer=optimizer,\\n        loss=tf.keras.losses.CategoricalCrossentropy(\\n            label_smoothing=experiment.model.label_smoothing if label_smoothing is None else label_smoothing,\\n            from_logits=True,\\n        ),\\n        metrics=[\\n            tf.keras.metrics.CategoricalAccuracy(name=\\"accuracy\\"),\\n            tf.keras.metrics.TopKCategoricalAccuracy(\\n                k=experiment.train.top_k,\\n                name=f\\"top{experiment.train.top_k}_accuracy\\",\\n            ),\\n        ],\\n    )\\n    return model\\n\\n\\ndef build_callbacks(output_dir: Path, patience: int) -> list[tf.keras.callbacks.Callback]:\\n    return build_callbacks_for_monitor(output_dir, patience=patience)\\n\\n\\ndef build_callbacks_for_monitor(\\n    output_dir: Path,\\n    patience: int,\\n    monitor: str = \\"val_accuracy\\",\\n    mode: str = \\"max\\",\\n    include_checkpoint: bool = True,\\n    save_weights_only: bool = False,\\n    csv_log_filename: str | None = \\"epoch_log.csv\\",\\n    min_delta: float = 0.0,\\n) -> list[tf.keras.callbacks.Callback]:\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    callbacks: list[tf.keras.callbacks.Callback] = [\\n        tf.keras.callbacks.EarlyStopping(\\n            monitor=monitor,\\n            mode=mode,\\n            patience=patience,\\n            min_delta=min_delta,\\n            restore_best_weights=True,\\n        ),\\n        tf.keras.callbacks.TerminateOnNaN(),\\n    ]\\n    if csv_log_filename:\\n        callbacks.append(\\n            tf.keras.callbacks.CSVLogger(\\n                str(output_dir / csv_log_filename),\\n                append=False,\\n            )\\n        )\\n    if include_checkpoint:\\n        filename = \\"best.weights.h5\\" if save_weights_only else \\"best.keras\\"\\n        callbacks.append(\\n            tf.keras.callbacks.ModelCheckpoint(\\n                filepath=str(output_dir / filename),\\n                monitor=monitor,\\n                mode=mode,\\n                save_best_only=True,\\n                save_weights_only=save_weights_only,\\n            )\\n        )\\n    return callbacks\\n\\n\\n@dataclass\\nclass HistoryBundle:\\n    history: dict[str, list[float]]\\n\\n\\ndef _merge_histories(stage_histories: list[tuple[str, tf.keras.callbacks.History | None]]) -> HistoryBundle | None:\\n    merged: dict[str, list[float]] = {}\\n    saw_history = False\\n    for _, history in stage_histories:\\n        if history is None:\\n            continue\\n        saw_history = True\\n        for key, values in history.history.items():\\n            merged.setdefault(key, []).extend(list(values))\\n    if not saw_history:\\n        return None\\n    return HistoryBundle(history=merged)\\n\\n\\ndef _student_only_dataset(dataset):\\n    element_spec = getattr(dataset, \\"element_spec\\", None)\\n    if not isinstance(element_spec, tuple) or len(element_spec) != 2:\\n        return dataset\\n    feature_spec, _ = element_spec\\n    if not isinstance(feature_spec, dict) or \\"student\\" not in feature_spec:\\n        return dataset\\n\\n    def select_student(features, labels):\\n        return features[\\"student\\"], labels\\n\\n    return dataset.map(select_student, num_parallel_calls=tf.data.AUTOTUNE)\\n\\n\\ndef _valid_pair(pair: tuple[str, str] | None) -> tuple[str, str] | None:\\n    if not pair or len(pair) != 2:\\n        return None\\n    if not pair[0] or not pair[1]:\\n        return None\\n    return pair\\n\\n\\ndef _valid_pairs(pairs: tuple[tuple[str, str], ...] | list[tuple[str, str]]) -> list[tuple[str, str]]:\\n    valid: list[tuple[str, str]] = []\\n    for pair in pairs:\\n        checked = _valid_pair(pair)\\n        if checked is not None:\\n            valid.append(checked)\\n    return valid\\n\\n\\ndef _unique_names(names: list[str]) -> list[str]:\\n    seen: set[str] = set()\\n    unique: list[str] = []\\n    for name in names:\\n        if name and name not in seen:\\n            seen.add(name)\\n            unique.append(name)\\n    return unique\\n\\n\\ndef _build_probe(model: tf.keras.Model, layer_names: list[str]) -> tf.keras.Model | None:\\n    if not layer_names:\\n        return None\\n    outputs = [model.output, *[model.get_layer(name).output for name in layer_names]]\\n    return tf.keras.Model(inputs=model.inputs, outputs=outputs, name=f\\"{model.name}_probe\\")\\n\\n\\ndef _unpack_probe_outputs(outputs, feature_names: list[str]) -> tuple[tf.Tensor, dict[str, tf.Tensor]]:\\n    if not feature_names:\\n        return outputs, {}\\n    if not isinstance(outputs, (list, tuple)):\\n        raise TypeError(\\"Probe outputs must be a list or tuple when feature names are provided.\\")\\n    logits = outputs[0]\\n    feature_values = outputs[1:]\\n    return logits, {name: tensor for name, tensor in zip(feature_names, feature_values)}\\n\\n\\ndef _align_teacher_map(student_map: tf.Tensor, teacher_map: tf.Tensor) -> tf.Tensor:\\n    student_hw = tf.shape(student_map)[1:3]\\n    return tf.image.resize(teacher_map, size=student_hw, method=\\"bilinear\\")\\n\\n\\ndef _normalize_feature_map(feature_map: tf.Tensor) -> tf.Tensor:\\n    return tf.nn.l2_normalize(feature_map, axis=-1)\\n\\n\\ndef _attention_vector(feature_map: tf.Tensor) -> tf.Tensor:\\n    attention = tf.reduce_mean(tf.square(feature_map), axis=-1)\\n    attention = tf.reshape(attention, [tf.shape(attention)[0], -1])\\n    return tf.nn.l2_normalize(attention, axis=-1)\\n\\n\\ndef _frame_similarity_matrix(feature_map: tf.Tensor) -> tf.Tensor:\\n    time_embeddings = tf.reduce_mean(feature_map, axis=2)\\n    time_embeddings = tf.nn.l2_normalize(time_embeddings, axis=-1)\\n    return tf.matmul(time_embeddings, time_embeddings, transpose_b=True)\\n\\n\\n@tf.keras.utils.register_keras_serializable(package=\\"kws_advanced\\")\\nclass Distiller(tf.keras.Model):\\n    def __init__(\\n        self,\\n        student: tf.keras.Model,\\n        teacher: tf.keras.Model,\\n        hint_layer_pair: tuple[str, str] | None = None,\\n        attention_layer_pairs: tuple[tuple[str, str], ...] | list[tuple[str, str]] = (),\\n        similarity_layer_pair: tuple[str, str] | None = None,\\n    ) -> None:\\n        super().__init__()\\n        self.student = student\\n        self.teacher = teacher\\n        self.hint_layer_pair = _valid_pair(hint_layer_pair)\\n        self.attention_layer_pairs = tuple(_valid_pairs(attention_layer_pairs))\\n        self.similarity_layer_pair = _valid_pair(similarity_layer_pair)\\n\\n        student_feature_names = []\\n        teacher_feature_names = []\\n        for student_name, teacher_name in self.attention_layer_pairs:\\n            student_feature_names.append(student_name)\\n            teacher_feature_names.append(teacher_name)\\n        if self.hint_layer_pair:\\n            student_feature_names.append(self.hint_layer_pair[0])\\n            teacher_feature_names.append(self.hint_layer_pair[1])\\n        if self.similarity_layer_pair:\\n            student_feature_names.append(self.similarity_layer_pair[0])\\n            teacher_feature_names.append(self.similarity_layer_pair[1])\\n\\n        self.student_feature_names = _unique_names(student_feature_names)\\n        self.teacher_feature_names = _unique_names(teacher_feature_names)\\n        self.student_probe = _build_probe(self.student, self.student_feature_names)\\n        self.teacher_probe = _build_probe(self.teacher, self.teacher_feature_names)\\n\\n        self.student_loss_fn: tf.keras.losses.Loss | None = None\\n        self.distillation_loss_fn: tf.keras.losses.Loss | None = None\\n        self.ce_weight = 0.2\\n        self.kd_weight = 0.8\\n        self.hint_weight = 0.0\\n        self.attention_weight = 0.0\\n        self.similarity_weight = 0.0\\n        self.temperature = 4.0\\n        self.metric_list: list[tf.keras.metrics.Metric] = []\\n\\n        self.loss_tracker = tf.keras.metrics.Mean(name=\\"loss\\")\\n        self.student_loss_tracker = tf.keras.metrics.Mean(name=\\"student_loss\\")\\n        self.distillation_loss_tracker = tf.keras.metrics.Mean(name=\\"distillation_loss\\")\\n        self.hint_loss_tracker = tf.keras.metrics.Mean(name=\\"hint_loss\\")\\n        self.attention_loss_tracker = tf.keras.metrics.Mean(name=\\"attention_loss\\")\\n        self.similarity_loss_tracker = tf.keras.metrics.Mean(name=\\"similarity_loss\\")\\n\\n        self.hint_projector: tf.keras.layers.Layer | None = None\\n        if self.hint_layer_pair:\\n            teacher_hint_layer = self.teacher.get_layer(self.hint_layer_pair[1])\\n            teacher_channels = int(teacher_hint_layer.output.shape[-1])\\n            self.hint_projector = tf.keras.layers.Conv2D(\\n                teacher_channels,\\n                kernel_size=(1, 1),\\n                padding=\\"same\\",\\n                use_bias=False,\\n                name=\\"hint_projector\\",\\n            )\\n\\n    @property\\n    def metrics(self):\\n        return [\\n            self.loss_tracker,\\n            self.student_loss_tracker,\\n            self.distillation_loss_tracker,\\n            self.hint_loss_tracker,\\n            self.attention_loss_tracker,\\n            self.similarity_loss_tracker,\\n            *self.metric_list,\\n        ]\\n\\n    def compile(\\n        self,\\n        optimizer: tf.keras.optimizers.Optimizer,\\n        metrics: list[tf.keras.metrics.Metric],\\n        student_loss_fn: tf.keras.losses.Loss,\\n        distillation_loss_fn: tf.keras.losses.Loss,\\n        ce_weight: float,\\n        kd_weight: float,\\n        hint_weight: float,\\n        attention_weight: float,\\n        similarity_weight: float,\\n        temperature: float,\\n    ) -> None:\\n        super().compile(optimizer=optimizer)\\n        self.student_loss_fn = student_loss_fn\\n        self.distillation_loss_fn = distillation_loss_fn\\n        self.ce_weight = ce_weight\\n        self.kd_weight = kd_weight\\n        self.hint_weight = hint_weight\\n        self.attention_weight = attention_weight\\n        self.similarity_weight = similarity_weight\\n        self.temperature = temperature\\n        self.metric_list = metrics\\n\\n    def call(self, inputs, training=False):\\n        student_features, _ = self._split_features(inputs)\\n        return self.student(student_features, training=training)\\n\\n    def get_config(self) -> dict:\\n        config = super().get_config()\\n        config.update(\\n            {\\n                \\"student\\": tf.keras.utils.serialize_keras_object(self.student),\\n                \\"teacher\\": tf.keras.utils.serialize_keras_object(self.teacher),\\n                \\"hint_layer_pair\\": self.hint_layer_pair,\\n                \\"attention_layer_pairs\\": list(self.attention_layer_pairs),\\n                \\"similarity_layer_pair\\": self.similarity_layer_pair,\\n            }\\n        )\\n        return config\\n\\n    @classmethod\\n    def from_config(cls, config: dict):\\n        config = dict(config)\\n        student = tf.keras.utils.deserialize_keras_object(config.pop(\\"student\\"))\\n        teacher = tf.keras.utils.deserialize_keras_object(config.pop(\\"teacher\\"))\\n        return cls(student=student, teacher=teacher, **config)\\n\\n    def _forward_student(\\n        self,\\n        features: tf.Tensor,\\n        training: bool,\\n    ) -> tuple[tf.Tensor, dict[str, tf.Tensor]]:\\n        if self.student_probe is None:\\n            logits = self.student(features, training=training)\\n            return logits, {}\\n        return _unpack_probe_outputs(self.student_probe(features, training=training), self.student_feature_names)\\n\\n    def _forward_teacher(self, features: tf.Tensor) -> tuple[tf.Tensor, dict[str, tf.Tensor]]:\\n        if self.teacher_probe is None:\\n            logits = self.teacher(features, training=False)\\n            return logits, {}\\n        return _unpack_probe_outputs(self.teacher_probe(features, training=False), self.teacher_feature_names)\\n\\n    def _split_features(self, features) -> tuple[tf.Tensor, tf.Tensor]:\\n        if isinstance(features, dict):\\n            student_features = features[\\"student\\"]\\n            teacher_features = features.get(\\"teacher\\", student_features)\\n            return student_features, teacher_features\\n        return features, features\\n\\n    def _compute_distillation_terms(\\n        self,\\n        labels: tf.Tensor,\\n        student_logits: tf.Tensor,\\n        teacher_logits: tf.Tensor,\\n        student_features: dict[str, tf.Tensor],\\n        teacher_features: dict[str, tf.Tensor],\\n        training: bool,\\n    ) -> tuple[tf.Tensor, dict[str, tf.Tensor]]:\\n        student_loss = self.student_loss_fn(labels, student_logits)\\n        zero = tf.constant(0.0, dtype=student_logits.dtype)\\n\\n        kd_loss = zero\\n        if self.kd_weight > 0.0:\\n            kd_loss = self.distillation_loss_fn(\\n                tf.nn.softmax(teacher_logits / self.temperature, axis=1),\\n                tf.nn.softmax(student_logits / self.temperature, axis=1),\\n            ) * (self.temperature ** 2)\\n\\n        hint_loss = zero\\n        if self.hint_weight > 0.0 and self.hint_layer_pair and self.hint_projector is not None:\\n            student_map = student_features[self.hint_layer_pair[0]]\\n            teacher_map = teacher_features[self.hint_layer_pair[1]]\\n            teacher_map = tf.stop_gradient(_align_teacher_map(student_map, teacher_map))\\n            projected_student = self.hint_projector(student_map, training=training)\\n            projected_student = _normalize_feature_map(projected_student)\\n            teacher_map = _normalize_feature_map(teacher_map)\\n            hint_loss = tf.reduce_mean(tf.square(projected_student - teacher_map))\\n\\n        attention_loss = zero\\n        if self.attention_weight > 0.0 and self.attention_layer_pairs:\\n            attention_terms = []\\n            for student_name, teacher_name in self.attention_layer_pairs:\\n                student_map = student_features[student_name]\\n                teacher_map = teacher_features[teacher_name]\\n                teacher_map = tf.stop_gradient(_align_teacher_map(student_map, teacher_map))\\n                student_attention = _attention_vector(student_map)\\n                teacher_attention = _attention_vector(teacher_map)\\n                attention_terms.append(tf.reduce_mean(tf.square(student_attention - teacher_attention)))\\n            attention_loss = tf.add_n(attention_terms) / float(len(attention_terms))\\n\\n        similarity_loss = zero\\n        if self.similarity_weight > 0.0 and self.similarity_layer_pair:\\n            student_map = student_features[self.similarity_layer_pair[0]]\\n            teacher_map = teacher_features[self.similarity_layer_pair[1]]\\n            teacher_map = tf.stop_gradient(_align_teacher_map(student_map, teacher_map))\\n            student_similarity = _frame_similarity_matrix(student_map)\\n            teacher_similarity = _frame_similarity_matrix(teacher_map)\\n            similarity_loss = tf.reduce_mean(tf.square(student_similarity - teacher_similarity))\\n\\n        total_loss = (\\n            self.ce_weight * student_loss\\n            + self.kd_weight * kd_loss\\n            + self.hint_weight * hint_loss\\n            + self.attention_weight * attention_loss\\n            + self.similarity_weight * similarity_loss\\n        )\\n\\n        return total_loss, {\\n            \\"student_loss\\": student_loss,\\n            \\"distillation_loss\\": kd_loss,\\n            \\"hint_loss\\": hint_loss,\\n            \\"attention_loss\\": attention_loss,\\n            \\"similarity_loss\\": similarity_loss,\\n        }\\n\\n    def train_step(self, data):\\n        features, labels = data\\n        student_inputs, teacher_inputs = self._split_features(features)\\n        teacher_logits, teacher_features = self._forward_teacher(teacher_inputs)\\n\\n        with tf.GradientTape() as tape:\\n            student_logits, student_features = self._forward_student(student_inputs, training=True)\\n            total_loss, losses = self._compute_distillation_terms(\\n                labels=labels,\\n                student_logits=student_logits,\\n                teacher_logits=teacher_logits,\\n                student_features=student_features,\\n                teacher_features=teacher_features,\\n                training=True,\\n            )\\n\\n        trainable_variables = self.trainable_variables\\n        gradients = tape.gradient(total_loss, trainable_variables)\\n        self.optimizer.apply_gradients(zip(gradients, trainable_variables))\\n\\n        self.loss_tracker.update_state(total_loss)\\n        self.student_loss_tracker.update_state(losses[\\"student_loss\\"])\\n        self.distillation_loss_tracker.update_state(losses[\\"distillation_loss\\"])\\n        self.hint_loss_tracker.update_state(losses[\\"hint_loss\\"])\\n        self.attention_loss_tracker.update_state(losses[\\"attention_loss\\"])\\n        self.similarity_loss_tracker.update_state(losses[\\"similarity_loss\\"])\\n        for metric in self.metric_list:\\n            metric.update_state(labels, student_logits)\\n        return {metric.name: metric.result() for metric in self.metrics}\\n\\n    def test_step(self, data):\\n        features, labels = data\\n        student_inputs, teacher_inputs = self._split_features(features)\\n        teacher_logits, teacher_features = self._forward_teacher(teacher_inputs)\\n        student_logits, student_features = self._forward_student(student_inputs, training=False)\\n        total_loss, losses = self._compute_distillation_terms(\\n            labels=labels,\\n            student_logits=student_logits,\\n            teacher_logits=teacher_logits,\\n            student_features=student_features,\\n            teacher_features=teacher_features,\\n            training=False,\\n        )\\n        self.loss_tracker.update_state(total_loss)\\n        self.student_loss_tracker.update_state(losses[\\"student_loss\\"])\\n        self.distillation_loss_tracker.update_state(losses[\\"distillation_loss\\"])\\n        self.hint_loss_tracker.update_state(losses[\\"hint_loss\\"])\\n        self.attention_loss_tracker.update_state(losses[\\"attention_loss\\"])\\n        self.similarity_loss_tracker.update_state(losses[\\"similarity_loss\\"])\\n        for metric in self.metric_list:\\n            metric.update_state(labels, student_logits)\\n        return {metric.name: metric.result() for metric in self.metrics}\\n\\n\\ndef build_distiller(\\n    student: tf.keras.Model,\\n    teacher: tf.keras.Model,\\n    experiment: ExperimentConfig,\\n    steps_per_epoch: int,\\n    stage: str = \\"main\\",\\n) -> Distiller:\\n    teacher.trainable = False\\n    train_cfg = experiment.train\\n\\n    hint_layer_pair: tuple[str, str] | None = None\\n    attention_layer_pairs: tuple[tuple[str, str], ...] = ()\\n    similarity_layer_pair: tuple[str, str] | None = None\\n    ce_weight = train_cfg.ce_loss_weight\\n    kd_weight = train_cfg.kd_loss_weight\\n    hint_weight = train_cfg.hint_loss_weight\\n    attention_weight = train_cfg.attention_loss_weight\\n    similarity_weight = train_cfg.similarity_loss_weight\\n    stage_epochs = train_cfg.student_epochs\\n\\n    if stage == \\"warmup\\":\\n        hint_layer_pair = _valid_pair(train_cfg.hint_layer_pair)\\n        ce_weight = train_cfg.hint_warmup_ce_weight\\n        kd_weight = 0.0\\n        hint_weight = train_cfg.hint_warmup_hint_weight\\n        attention_weight = 0.0\\n        similarity_weight = 0.0\\n        stage_epochs = train_cfg.hint_warmup_epochs\\n    elif stage == \\"main\\":\\n        if train_cfg.distillation_mode == \\"multilevel\\":\\n            hint_layer_pair = _valid_pair(train_cfg.hint_layer_pair)\\n            attention_layer_pairs = tuple(_valid_pairs(train_cfg.attention_layer_pairs))\\n            similarity_layer_pair = _valid_pair(train_cfg.similarity_layer_pair)\\n    else:\\n        raise KeyError(f\\"Unsupported distillation stage: {stage}\\")\\n\\n    distiller = Distiller(\\n        student=student,\\n        teacher=teacher,\\n        hint_layer_pair=hint_layer_pair,\\n        attention_layer_pairs=attention_layer_pairs,\\n        similarity_layer_pair=similarity_layer_pair,\\n    )\\n    distiller.compile(\\n        optimizer=_build_optimizer(\\n            experiment=experiment,\\n            learning_rate=experiment.train.student_lr,\\n            total_steps=stage_epochs * steps_per_epoch,\\n        ),\\n        metrics=[\\n            tf.keras.metrics.CategoricalAccuracy(name=\\"accuracy\\"),\\n            tf.keras.metrics.TopKCategoricalAccuracy(\\n                k=experiment.train.top_k,\\n                name=f\\"top{experiment.train.top_k}_accuracy\\",\\n            ),\\n        ],\\n        student_loss_fn=tf.keras.losses.CategoricalCrossentropy(\\n            label_smoothing=experiment.model.label_smoothing,\\n            from_logits=True,\\n        ),\\n        distillation_loss_fn=tf.keras.losses.KLDivergence(),\\n        ce_weight=ce_weight,\\n        kd_weight=kd_weight,\\n        hint_weight=hint_weight,\\n        attention_weight=attention_weight,\\n        similarity_weight=similarity_weight,\\n        temperature=experiment.train.distill_temperature,\\n    )\\n    return distiller\\n\\n\\ndef train_student_model(\\n    student: tf.keras.Model,\\n    teacher: tf.keras.Model,\\n    train_ds,\\n    val_ds,\\n    experiment: ExperimentConfig,\\n    steps_per_epoch: int,\\n    output_dir: Path,\\n) -> dict[str, object]:\\n    train_cfg = experiment.train\\n    stage_histories: list[tuple[str, tf.keras.callbacks.History | None]] = []\\n    stage_details: list[dict[str, object]] = []\\n    student_train_ds = _student_only_dataset(train_ds)\\n    student_val_ds = _student_only_dataset(val_ds)\\n\\n    if train_cfg.student_pretrain_epochs > 0:\\n        pretrain_dir = output_dir / \\"pretrain\\"\\n        student = compile_classifier(\\n            student,\\n            experiment=experiment,\\n            learning_rate=train_cfg.student_lr,\\n            epochs=train_cfg.student_pretrain_epochs,\\n            steps_per_epoch=steps_per_epoch,\\n        )\\n        pretrain_history = student.fit(\\n            student_train_ds,\\n            validation_data=student_val_ds,\\n            epochs=train_cfg.student_pretrain_epochs,\\n            callbacks=build_callbacks_for_monitor(\\n                pretrain_dir,\\n                patience=max(2, train_cfg.early_stopping_patience),\\n                monitor=\\"val_accuracy\\",\\n                mode=\\"max\\",\\n                include_checkpoint=True,\\n                min_delta=train_cfg.early_stopping_min_delta,\\n            ),\\n        )\\n        stage_histories.append((\\"pretrain\\", pretrain_history))\\n        stage_details.append(\\n            {\\n                \\"name\\": \\"pretrain\\",\\n                \\"epochs_requested\\": train_cfg.student_pretrain_epochs,\\n                \\"epoch_log_path\\": str(pretrain_dir / \\"epoch_log.csv\\"),\\n                \\"checkpoint_path\\": str(pretrain_dir / \\"best.keras\\"),\\n            }\\n        )\\n\\n    if train_cfg.uses_distillation:\\n        if train_cfg.distillation_mode == \\"multilevel\\" and train_cfg.hint_warmup_epochs > 0 and train_cfg.hint_warmup_hint_weight > 0.0:\\n            warmup_dir = output_dir / \\"hint_warmup\\"\\n            warmup_distiller = build_distiller(\\n                student=student,\\n                teacher=teacher,\\n                experiment=experiment,\\n                steps_per_epoch=steps_per_epoch,\\n                stage=\\"warmup\\",\\n            )\\n            warmup_history = warmup_distiller.fit(\\n                train_ds,\\n                validation_data=val_ds,\\n                epochs=train_cfg.hint_warmup_epochs,\\n                callbacks=build_callbacks_for_monitor(\\n                    warmup_dir,\\n                    patience=1,\\n                    monitor=\\"val_accuracy\\",\\n                    mode=\\"max\\",\\n                    include_checkpoint=False,\\n                    min_delta=train_cfg.early_stopping_min_delta,\\n                ),\\n            )\\n            student = warmup_distiller.student\\n            stage_histories.append((\\"hint_warmup\\", warmup_history))\\n            stage_details.append(\\n                {\\n                    \\"name\\": \\"hint_warmup\\",\\n                    \\"epochs_requested\\": train_cfg.hint_warmup_epochs,\\n                    \\"epoch_log_path\\": str(warmup_dir / \\"epoch_log.csv\\"),\\n                }\\n            )\\n\\n        distiller = build_distiller(\\n            student=student,\\n            teacher=teacher,\\n            experiment=experiment,\\n            steps_per_epoch=steps_per_epoch,\\n            stage=\\"main\\",\\n        )\\n        distill_history = distiller.fit(\\n            train_ds,\\n            validation_data=val_ds,\\n            epochs=train_cfg.student_epochs,\\n            callbacks=build_callbacks_for_monitor(\\n                output_dir,\\n                patience=train_cfg.early_stopping_patience,\\n                monitor=\\"val_accuracy\\",\\n                mode=\\"max\\",\\n                include_checkpoint=False,\\n                min_delta=train_cfg.early_stopping_min_delta,\\n            ),\\n        )\\n        student = distiller.student\\n        stage_histories.append((\\"distillation\\", distill_history))\\n        stage_details.append(\\n            {\\n                \\"name\\": \\"distillation\\",\\n                \\"epochs_requested\\": train_cfg.student_epochs,\\n                \\"epoch_log_path\\": str(output_dir / \\"epoch_log.csv\\"),\\n                \\"mode\\": train_cfg.distillation_mode,\\n            }\\n        )\\n    else:\\n        student = compile_classifier(\\n            student,\\n            experiment=experiment,\\n            learning_rate=train_cfg.student_lr,\\n            epochs=train_cfg.student_epochs,\\n            steps_per_epoch=steps_per_epoch,\\n        )\\n        history = student.fit(\\n            student_train_ds,\\n            validation_data=student_val_ds,\\n            epochs=train_cfg.student_epochs,\\n            callbacks=build_callbacks_for_monitor(\\n                output_dir,\\n                patience=train_cfg.early_stopping_patience,\\n                monitor=\\"val_accuracy\\",\\n                mode=\\"max\\",\\n                include_checkpoint=True,\\n                min_delta=train_cfg.early_stopping_min_delta,\\n            ),\\n        )\\n        stage_histories.append((\\"student\\", history))\\n        stage_details.append(\\n            {\\n                \\"name\\": \\"student\\",\\n                \\"epochs_requested\\": train_cfg.student_epochs,\\n                \\"epoch_log_path\\": str(output_dir / \\"epoch_log.csv\\"),\\n                \\"checkpoint_path\\": str(output_dir / \\"best.keras\\"),\\n            }\\n        )\\n\\n    if train_cfg.student_polish_epochs > 0:\\n        polish_dir = output_dir / \\"polish\\"\\n        student = compile_classifier(\\n            student,\\n            experiment=experiment,\\n            learning_rate=train_cfg.student_lr * 0.25,\\n            epochs=train_cfg.student_polish_epochs,\\n            steps_per_epoch=max(1, steps_per_epoch),\\n        )\\n        polish_history = student.fit(\\n            student_train_ds,\\n            validation_data=student_val_ds,\\n            epochs=train_cfg.student_polish_epochs,\\n            callbacks=build_callbacks_for_monitor(\\n                polish_dir,\\n                patience=1,\\n                monitor=\\"val_accuracy\\",\\n                mode=\\"max\\",\\n                include_checkpoint=False,\\n                min_delta=train_cfg.early_stopping_min_delta,\\n            ),\\n        )\\n        stage_histories.append((\\"polish\\", polish_history))\\n        stage_details.append(\\n            {\\n                \\"name\\": \\"polish\\",\\n                \\"epochs_requested\\": train_cfg.student_polish_epochs,\\n                \\"epoch_log_path\\": str(polish_dir / \\"epoch_log.csv\\"),\\n            }\\n        )\\n\\n    merged_history = _merge_histories(stage_histories)\\n    return {\\n        \\"student\\": student,\\n        \\"history\\": merged_history,\\n        \\"extra\\": {\\n            \\"distillation_enabled\\": bool(train_cfg.uses_distillation),\\n            \\"distillation_mode\\": train_cfg.distillation_mode if train_cfg.uses_distillation else \\"none\\",\\n            \\"training_stages\\": stage_details,\\n            \\"main_epoch_log_path\\": str(output_dir / \\"epoch_log.csv\\"),\\n            \\"checkpoint_path\\": \\"\\" if train_cfg.uses_distillation else str(output_dir / \\"best.keras\\"),\\n        },\\n    }\\n\\n\\ndef maybe_quantize_aware_clone(model: tf.keras.Model) -> tf.keras.Model | None:\\n    try:\\n        import tensorflow_model_optimization as tfmot\\n    except Exception:\\n        return None\\n    return tfmot.quantization.keras.quantize_model(model)\\n", "code/training/kws_advanced/export.py": "from __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport tensorflow as tf\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef representative_dataset(\\n    dataset: tf.data.Dataset,\\n    batch_limit: int,\\n):\\n    for batch_features, _ in dataset.take(batch_limit):\\n        for item in batch_features:\\n            yield [tf.expand_dims(tf.cast(item, tf.float32), axis=0)]\\n\\n\\ndef convert_to_int8_tflite(\\n    model: tf.keras.Model,\\n    representative_data: tf.data.Dataset,\\n    output_path: Path,\\n    representative_batches: int,\\n) -> bytes:\\n    converter = tf.lite.TFLiteConverter.from_keras_model(model)\\n    converter.optimizations = [tf.lite.Optimize.DEFAULT]\\n    converter.representative_dataset = lambda: representative_dataset(\\n        representative_data,\\n        batch_limit=representative_batches,\\n    )\\n    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]\\n    converter.inference_input_type = tf.int8\\n    converter.inference_output_type = tf.int8\\n    tflite_bytes = converter.convert()\\n    output_path.parent.mkdir(parents=True, exist_ok=True)\\n    output_path.write_bytes(tflite_bytes)\\n    return tflite_bytes\\n\\n\\ndef inspect_tflite_model(model_path: Path) -> dict[str, Any]:\\n    interpreter = tf.lite.Interpreter(model_path=str(model_path))\\n    interpreter.allocate_tensors()\\n    input_details = interpreter.get_input_details()[0]\\n    output_details = interpreter.get_output_details()[0]\\n    input_scale, input_zero_point = input_details[\\"quantization\\"]\\n    output_scale, output_zero_point = output_details[\\"quantization\\"]\\n    return {\\n        \\"input_shape\\": input_details[\\"shape\\"].tolist(),\\n        \\"input_dtype\\": str(input_details[\\"dtype\\"]),\\n        \\"input_scale\\": float(input_scale),\\n        \\"input_zero_point\\": int(input_zero_point),\\n        \\"output_shape\\": output_details[\\"shape\\"].tolist(),\\n        \\"output_dtype\\": str(output_details[\\"dtype\\"]),\\n        \\"output_scale\\": float(output_scale),\\n        \\"output_zero_point\\": int(output_zero_point),\\n        \\"tensor_count\\": len(interpreter.get_tensor_details()),\\n    }\\n\\n\\ndef write_c_array(\\n    tflite_bytes: bytes,\\n    output_dir: Path,\\n    array_name: str,\\n) -> dict[str, Path]:\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    header_path = output_dir / f\\"{array_name}.h\\"\\n    source_path = output_dir / f\\"{array_name}.cc\\"\\n    include_guard = f\\"{array_name.upper()}_H_\\"\\n\\n    header_text = \\"\\\\n\\".join(\\n        [\\n            f\\"#ifndef {include_guard}\\",\\n            f\\"#define {include_guard}\\",\\n            \\"\\",\\n            \\"#include <cstddef>\\",\\n            \\"#include <cstdint>\\",\\n            \\"\\",\\n            f\\"extern const unsigned char {array_name}[];\\",\\n            f\\"extern const unsigned int {array_name}_len;\\",\\n            \\"\\",\\n            f\\"#endif  // {include_guard}\\",\\n            \\"\\",\\n        ]\\n    )\\n\\n    hex_lines = []\\n    for offset in range(0, len(tflite_bytes), 12):\\n        chunk = tflite_bytes[offset : offset + 12]\\n        hex_lines.append(\\"  \\" + \\", \\".join(f\\"0x{byte:02x}\\" for byte in chunk))\\n\\n    source_text = \\"\\\\n\\".join(\\n        [\\n            f\'#include \\"{header_path.name}\\"\',\\n            \\"\\",\\n            f\\"const unsigned char {array_name}[] = {{\\",\\n            \\",\\\\n\\".join(hex_lines),\\n            \\"};\\",\\n            f\\"const unsigned int {array_name}_len = {len(tflite_bytes)};\\",\\n            \\"\\",\\n        ]\\n    )\\n\\n    header_path.write_text(header_text, encoding=\\"utf-8\\")\\n    source_path.write_text(source_text, encoding=\\"utf-8\\")\\n    return {\\"header\\": header_path, \\"source\\": source_path}\\n\\n\\ndef export_experiment_artifacts(\\n    model: tf.keras.Model,\\n    representative_data: tf.data.Dataset,\\n    experiment: ExperimentConfig,\\n    required_ops: list[str],\\n) -> dict[str, Any]:\\n    artifact_dir = Path(experiment.export.artifacts_dir) / experiment.tag\\n    model_path = artifact_dir / f\\"{experiment.export.model_stem}.tflite\\"\\n    tflite_bytes = convert_to_int8_tflite(\\n        model,\\n        representative_data=representative_data,\\n        output_path=model_path,\\n        representative_batches=experiment.export.representative_batches,\\n    )\\n    tflite_info = inspect_tflite_model(model_path)\\n\\n    c_paths: dict[str, str] = {}\\n    if experiment.export.export_c_array:\\n        array_name = f\\"{experiment.export.model_stem}_model\\"\\n        written = write_c_array(tflite_bytes, output_dir=artifact_dir, array_name=array_name)\\n        c_paths = {key: str(path) for key, path in written.items()}\\n\\n    metadata = {\\n        \\"experiment\\": experiment.to_dict(),\\n        \\"model_size_bytes\\": len(tflite_bytes),\\n        \\"required_tflite_ops\\": required_ops,\\n        \\"tflite\\": tflite_info,\\n        \\"labels\\": experiment.all_labels,\\n        \\"artifacts\\": {\\n            \\"tflite\\": str(model_path),\\n            **c_paths,\\n        },\\n    }\\n    metadata_path = artifact_dir / \\"metadata.json\\"\\n    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return metadata\\n", "code/training/kws_advanced/reporting.py": "from __future__ import annotations\\n\\nimport json\\nfrom copy import deepcopy\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport matplotlib.pyplot as plt\\nimport tensorflow as tf\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef get_run_dir(project_root: Path, experiment: ExperimentConfig) -> Path:\\n    run_dir = project_root / \\"code\\" / \\"training\\" / \\"runs\\" / experiment.tag\\n    run_dir.mkdir(parents=True, exist_ok=True)\\n    return run_dir\\n\\n\\ndef _state_path(run_dir: Path) -> Path:\\n    return run_dir / \\"run_state.json\\"\\n\\n\\ndef _summary_path(run_dir: Path) -> Path:\\n    return run_dir / \\"run_summary.md\\"\\n\\n\\ndef _read_state(run_dir: Path) -> dict[str, Any]:\\n    path = _state_path(run_dir)\\n    if not path.exists():\\n        return {}\\n    return json.loads(path.read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef _write_state(run_dir: Path, state: dict[str, Any]) -> None:\\n    _state_path(run_dir).write_text(\\n        json.dumps(state, ensure_ascii=False, indent=2),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n\\ndef _merge_dict(base: dict[str, Any], update: dict[str, Any]) -> dict[str, Any]:\\n    merged = deepcopy(base)\\n    for key, value in update.items():\\n        if isinstance(value, dict) and isinstance(merged.get(key), dict):\\n            merged[key] = _merge_dict(merged[key], value)\\n        else:\\n            merged[key] = value\\n    return merged\\n\\n\\ndef initialize_run_state(\\n    project_root: Path,\\n    experiment: ExperimentConfig,\\n    recipe_name: str,\\n    dataset_summary: dict[str, Any] | None = None,\\n) -> Path:\\n    run_dir = get_run_dir(project_root, experiment)\\n    state = _merge_dict(\\n        _read_state(run_dir),\\n        {\\n            \\"experiment\\": experiment.to_dict(),\\n            \\"recipe_name\\": recipe_name,\\n            \\"dataset_summary\\": dataset_summary or {},\\n            \\"stages\\": {},\\n            \\"artifacts\\": {},\\n            \\"notes\\": [],\\n        },\\n    )\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n    return run_dir\\n\\n\\ndef save_json_artifact(run_dir: Path, name: str, payload: Any) -> Path:\\n    path = run_dir / name\\n    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return path\\n\\n\\ndef save_text_artifact(run_dir: Path, name: str, text: str) -> Path:\\n    path = run_dir / name\\n    path.write_text(text, encoding=\\"utf-8\\")\\n    return path\\n\\n\\ndef save_current_figure(run_dir: Path, name: str) -> Path:\\n    path = run_dir / name\\n    plt.savefig(path, bbox_inches=\\"tight\\")\\n    return path\\n\\n\\ndef save_model_summary(run_dir: Path, model: tf.keras.Model, stage_name: str) -> Path:\\n    lines: list[str] = []\\n    model.summary(print_fn=lines.append)\\n    return save_text_artifact(run_dir, f\\"{stage_name}_model_summary.txt\\", \\"\\\\n\\".join(lines) + \\"\\\\n\\")\\n\\n\\ndef save_history(run_dir: Path, history, stage_name: str) -> Path | None:\\n    if history is None:\\n        return None\\n    return save_json_artifact(run_dir, f\\"{stage_name}_history.json\\", history.history)\\n\\n\\ndef save_metrics(run_dir: Path, metrics: dict[str, Any], stage_name: str) -> Path:\\n    return save_json_artifact(run_dir, f\\"{stage_name}_metrics.json\\", metrics)\\n\\n\\ndef save_history_plots(run_dir: Path, history, stage_name: str) -> list[str]:\\n    if history is None:\\n        return []\\n    hist = history.history\\n    if not hist:\\n        return []\\n\\n    plot_paths: list[str] = []\\n    groups = {\\n        \\"loss\\": [key for key in hist if \\"loss\\" in key],\\n        \\"metrics\\": [\\n            key\\n            for key in hist\\n            if \\"accuracy\\" in key or key.startswith(\\"top\\")\\n        ],\\n    }\\n    for suffix, keys in groups.items():\\n        if not keys:\\n            continue\\n        plt.figure(figsize=(10, 4))\\n        for key in keys:\\n            plt.plot(hist[key], label=key)\\n        plt.title(f\\"{stage_name} {suffix}\\")\\n        plt.grid(True, alpha=0.3)\\n        plt.legend()\\n        path = run_dir / f\\"{stage_name}_{suffix}.png\\"\\n        plt.savefig(path, bbox_inches=\\"tight\\")\\n        plt.close()\\n        plot_paths.append(str(path))\\n    return plot_paths\\n\\n\\ndef update_stage_state(\\n    run_dir: Path,\\n    stage_name: str,\\n    metrics: dict[str, Any] | None = None,\\n    history_path: str | None = None,\\n    plot_paths: list[str] | None = None,\\n    summary_path: str | None = None,\\n    extra: dict[str, Any] | None = None,\\n) -> None:\\n    state = _read_state(run_dir)\\n    stage_block = state.setdefault(\\"stages\\", {}).get(stage_name, {})\\n    update = {\\n        \\"metrics\\": metrics or stage_block.get(\\"metrics\\", {}),\\n        \\"history_path\\": history_path or stage_block.get(\\"history_path\\", \\"\\"),\\n        \\"plot_paths\\": plot_paths or stage_block.get(\\"plot_paths\\", []),\\n        \\"summary_path\\": summary_path or stage_block.get(\\"summary_path\\", \\"\\"),\\n    }\\n    if extra:\\n        update[\\"extra\\"] = _merge_dict(stage_block.get(\\"extra\\", {}), extra)\\n    state.setdefault(\\"stages\\", {})[stage_name] = _merge_dict(stage_block, update)\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef add_note(run_dir: Path, title: str, body: str) -> None:\\n    state = _read_state(run_dir)\\n    state.setdefault(\\"notes\\", []).append({\\"title\\": title, \\"body\\": body})\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef record_export_artifacts(run_dir: Path, metadata: dict[str, Any]) -> None:\\n    state = _read_state(run_dir)\\n    state[\\"artifacts\\"] = _merge_dict(state.get(\\"artifacts\\", {}), metadata)\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef write_run_summary(run_dir: Path) -> Path:\\n    state = _read_state(run_dir)\\n    lines = [\\n        f\\"# Run Summary: {state.get(\'experiment\', {}).get(\'tag\', run_dir.name)}\\",\\n        \\"\\",\\n        f\\"Recipe: `{state.get(\'recipe_name\', \'\')}`\\",\\n        \\"\\",\\n        \\"## Experiment\\",\\n        \\"\\",\\n        \\"```json\\",\\n        json.dumps(state.get(\\"experiment\\", {}), ensure_ascii=False, indent=2),\\n        \\"```\\",\\n        \\"\\",\\n        \\"## Dataset\\",\\n        \\"\\",\\n        \\"```json\\",\\n        json.dumps(state.get(\\"dataset_summary\\", {}), ensure_ascii=False, indent=2),\\n        \\"```\\",\\n        \\"\\",\\n        \\"## Stages\\",\\n        \\"\\",\\n    ]\\n\\n    for stage_name, payload in state.get(\\"stages\\", {}).items():\\n        lines.extend(\\n            [\\n                f\\"### {stage_name}\\",\\n                \\"\\",\\n                \\"```json\\",\\n                json.dumps(payload.get(\\"metrics\\", {}), ensure_ascii=False, indent=2),\\n                \\"```\\",\\n                \\"\\",\\n            ]\\n        )\\n        if payload.get(\\"history_path\\"):\\n            lines.append(f\\"History: `{payload[\'history_path\']}`\\")\\n        if payload.get(\\"summary_path\\"):\\n            lines.append(f\\"Model summary: `{payload[\'summary_path\']}`\\")\\n        for plot_path in payload.get(\\"plot_paths\\", []):\\n            lines.append(f\\"Plot: `{plot_path}`\\")\\n        if payload.get(\\"extra\\"):\\n            lines.extend(\\n                [\\n                    \\"Extra:\\",\\n                    \\"```json\\",\\n                    json.dumps(payload[\\"extra\\"], ensure_ascii=False, indent=2),\\n                    \\"```\\",\\n                ]\\n            )\\n        lines.append(\\"\\")\\n\\n    if state.get(\\"artifacts\\"):\\n        lines.extend(\\n            [\\n                \\"## Artifacts\\",\\n                \\"\\",\\n                \\"```json\\",\\n                json.dumps(state[\\"artifacts\\"], ensure_ascii=False, indent=2),\\n                \\"```\\",\\n                \\"\\",\\n            ]\\n        )\\n\\n    if state.get(\\"notes\\"):\\n        lines.extend([\\"## Notes\\", \\"\\"])\\n        for note in state[\\"notes\\"]:\\n            lines.append(f\\"### {note.get(\'title\', \'note\')}\\")\\n            lines.append(\\"\\")\\n            lines.append(note.get(\\"body\\", \\"\\"))\\n            lines.append(\\"\\")\\n\\n    summary_path = _summary_path(run_dir)\\n    summary_path.write_text(\\"\\\\n\\".join(lines), encoding=\\"utf-8\\")\\n    return summary_path\\n"}')

def ensure_runtime_files(root: Path, payloads: dict[str, str], overwrite: bool = False):
    created = []
    skipped = []
    for relative_path, content in payloads.items():
        target_path = root / relative_path
        target_path.parent.mkdir(parents=True, exist_ok=True)
        if target_path.exists() and not overwrite:
            skipped.append(relative_path)
            continue
        target_path.write_text(content, encoding="utf-8")
        created.append(relative_path)
    return created, skipped

created_files, skipped_files = ensure_runtime_files(
    PROJECT_ROOT,
    FILE_PAYLOADS,
    overwrite=FORCE_SYNC_RUNTIME_FILES,
)

if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "kws_advanced" or module_name.startswith("kws_advanced."):
        del sys.modules[module_name]

print("Runtime project root:", PROJECT_ROOT)
print("Runtime training root:", TRAINING_ROOT)
print("Created files:", len(created_files))
print("Skipped existing files:", len(skipped_files))
print("Training package exists:", (TRAINING_ROOT / "kws_advanced").is_dir())
print("Runtime source sync mode: force overwrite")


In [ ]:
import json

import matplotlib.pyplot as plt
import tensorflow as tf

from kws_advanced.config import make_experiment
from kws_advanced.data import prepare_datasets
from kws_advanced.distillation import (
    build_callbacks_for_monitor,
    compile_classifier,
    maybe_quantize_aware_clone,
    train_student_model,
)
from kws_advanced.experiments import build_recipe_book
from kws_advanced.export import export_experiment_artifacts
from kws_advanced.models import (
    build_student_model,
    build_teacher_model,
    estimate_maccs,
    required_tflite_ops,
)
from kws_advanced.reporting import (
    add_note,
    initialize_run_state,
    record_export_artifacts,
    save_current_figure,
    save_history,
    save_history_plots,
    save_json_artifact,
    save_metrics,
    save_model_summary,
    save_text_artifact,
    update_stage_state,
    write_run_summary,
)

tf.keras.utils.set_random_seed(42)
print("Project root:", PROJECT_ROOT)
print("Training root:", TRAINING_ROOT)


In [ ]:
base_experiment = make_experiment(tag="kws12_iterlab_v1", vocabulary_preset="kws12")
recipe_book = build_recipe_book(base_experiment)
print("Available recipes:", list(recipe_book))

SELECTED_RECIPE = "distilled_student_v3_bcresnet_teacher_seq"
experiment = recipe_book[SELECTED_RECIPE]
RUN_NOTES = ""
print(json.dumps(experiment.to_dict(), indent=2))


In [ ]:
AUTO_MOUNT_DRIVE_FOR_TEACHER = True
REUSE_TEACHER_CHECKPOINT = True
TEACHER_CHECKPOINT_PATH = ""
AUTO_FIND_LATEST_TEACHER_CHECKPOINT = True
TEACHER_SEARCH_ROOT = "/content/drive/MyDrive/diploma_esp32_teacher_saves"

def maybe_mount_drive_for_teacher() -> None:
    if not AUTO_MOUNT_DRIVE_FOR_TEACHER:
        return
    drive_root = Path("/content/drive")
    if drive_root.exists() and (drive_root / "MyDrive").exists():
        return
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Google Drive mount skipped:", exc)

def resolve_teacher_checkpoint() -> Path | None:
    candidate = TEACHER_CHECKPOINT_PATH.strip()
    if candidate:
        path = Path(candidate)
        return path if path.exists() else None
    if not AUTO_FIND_LATEST_TEACHER_CHECKPOINT:
        return None
    search_root = Path(TEACHER_SEARCH_ROOT)
    if not search_root.exists():
        return None
    teacher_tag = experiment.teacher_reuse_tag or experiment.tag
    matches = sorted(
        search_root.glob(f"{teacher_tag}_*/teacher_best.keras"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    return matches[0] if matches else None

maybe_mount_drive_for_teacher()
resolved_teacher_checkpoint = resolve_teacher_checkpoint()
print("Teacher reuse enabled:", REUSE_TEACHER_CHECKPOINT)
print("Resolved teacher checkpoint:", resolved_teacher_checkpoint)


In [ ]:
import os

# Optional dataset overrides for SSL-restricted or offline runtimes.
# Set only one of these if automatic download fails.
SPEECH_COMMANDS_DATASET_ROOT = ""
SPEECH_COMMANDS_ARCHIVE_PATH = ""

if SPEECH_COMMANDS_DATASET_ROOT:
    os.environ["SPEECH_COMMANDS_DATASET_ROOT"] = SPEECH_COMMANDS_DATASET_ROOT
if SPEECH_COMMANDS_ARCHIVE_PATH:
    os.environ["SPEECH_COMMANDS_ARCHIVE_PATH"] = SPEECH_COMMANDS_ARCHIVE_PATH

print("SPEECH_COMMANDS_DATASET_ROOT =", os.environ.get("SPEECH_COMMANDS_DATASET_ROOT", ""))
print("SPEECH_COMMANDS_ARCHIVE_PATH =", os.environ.get("SPEECH_COMMANDS_ARCHIVE_PATH", ""))


In [ ]:
import subprocess
import tarfile

DATA_DIR = PROJECT_ROOT / "data"
ARCHIVE_PATH = DATA_DIR / "speech_commands_v0.02.tar.gz"
EXTRACTED_DIR = DATA_DIR / "speech_commands_v0.02"
SOURCE_URL = "https://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"

def dataset_ready(root: Path) -> bool:
    required = [
        root / "validation_list.txt",
        root / "testing_list.txt",
        root / "yes",
        root / "no",
    ]
    return all(path.exists() for path in required)

DATA_DIR.mkdir(parents=True, exist_ok=True)

if dataset_ready(EXTRACTED_DIR):
    print("Dataset already prepared:", EXTRACTED_DIR)
else:
    if not ARCHIVE_PATH.exists():
        commands = [
            ["bash", "-lc", f"wget --no-check-certificate -O '{ARCHIVE_PATH.as_posix()}' '{SOURCE_URL}'"],
            ["bash", "-lc", f"curl -L -k '{SOURCE_URL}' -o '{ARCHIVE_PATH.as_posix()}'"],
        ]
        last_error = None
        for command in commands:
            try:
                print("Trying:", " ".join(command[:2]), "...")
                subprocess.run(command, check=True)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
        if last_error is not None:
            raise RuntimeError(f"Failed to download dataset archive: {last_error}")

    EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(path=EXTRACTED_DIR)

    if not dataset_ready(EXTRACTED_DIR):
        raise RuntimeError(
            "Extraction finished, but the dataset structure is incomplete: "
            f"{EXTRACTED_DIR}"
        )

    print("Dataset prepared:", EXTRACTED_DIR)


In [ ]:
bundle = prepare_datasets(PROJECT_ROOT, experiment)
teacher_train_ds = bundle["datasets"]["teacher_train"]
teacher_val_ds = bundle["datasets"]["teacher_validation"]
teacher_test_ds = bundle["datasets"]["teacher_test"]
student_train_ds = bundle["datasets"]["student_train"]
student_val_ds = bundle["datasets"]["student_validation"]
student_test_ds = bundle["datasets"]["student_test"]
distillation_train_ds = bundle["datasets"]["distillation_train"]
distillation_val_ds = bundle["datasets"]["distillation_validation"]
steps_per_epoch = bundle["steps_per_epoch"]

run_dir = initialize_run_state(
    PROJECT_ROOT,
    experiment,
    recipe_name=SELECTED_RECIPE,
    dataset_summary=bundle["summary"],
)
teacher_dir = run_dir / "teacher"
student_dir = run_dir / "student"
qat_dir = run_dir / "qat"
save_json_artifact(run_dir, "experiment.json", experiment.to_dict())
save_json_artifact(run_dir, "dataset_summary.json", bundle["summary"])
save_json_artifact(run_dir, "frontend_runtime.json", bundle["frontend"])
save_text_artifact(run_dir, "selected_recipe.txt", SELECTED_RECIPE + "\n")
if RUN_NOTES.strip():
    save_text_artifact(run_dir, "run_notes.txt", RUN_NOTES.strip() + "\n")
    add_note(run_dir, "Operator notes", RUN_NOTES.strip())

print("Dataset root:", bundle["dataset_root"])
print("Steps per epoch:", steps_per_epoch)
print("Run dir:", run_dir)
print(json.dumps(bundle["summary"], indent=2))
print(json.dumps(bundle["frontend"], indent=2))


In [ ]:
batch_features, batch_labels = next(iter(student_train_ds.take(1)))
print("Student feature batch shape:", batch_features.shape)
print("Label batch shape:", batch_labels.shape)

plt.figure(figsize=(10, 4))
plt.imshow(batch_features[0, :, :, 0].numpy().T, aspect="auto", origin="lower")
plt.title("Example student training feature map")
plt.xlabel("Frames")
plt.ylabel("Channels")
plt.colorbar()
preview_path = save_current_figure(run_dir, "student_feature_map_preview.png")
print("Saved preview:", preview_path)
plt.show()
plt.close()

teacher_preview_features, _ = next(iter(teacher_train_ds.take(1)))
if tuple(teacher_preview_features.shape[1:]) != tuple(batch_features.shape[1:]):
    plt.figure(figsize=(10, 4))
    plt.imshow(teacher_preview_features[0, :, :, 0].numpy().T, aspect="auto", origin="lower")
    plt.title("Example teacher training feature map")
    plt.xlabel("Frames")
    plt.ylabel("Channels")
    plt.colorbar()
    teacher_preview_path = save_current_figure(run_dir, "teacher_feature_map_preview.png")
    print("Saved teacher preview:", teacher_preview_path)
    plt.show()
    plt.close()


In [ ]:
teacher = build_teacher_model(experiment)
student = build_student_model(experiment)

teacher_params = teacher.count_params()
teacher_maccs = estimate_maccs(teacher)
student_params = student.count_params()
student_maccs = estimate_maccs(student)

teacher_summary_path = save_model_summary(run_dir, teacher, "teacher")
student_summary_path = save_model_summary(run_dir, student, "student")
model_inventory = {
    "teacher": {
        "params": teacher_params,
        "maccs_rough": teacher_maccs,
        "summary_path": str(teacher_summary_path),
    },
    "student": {
        "params": student_params,
        "maccs_rough": student_maccs,
        "summary_path": str(student_summary_path),
    },
}
save_json_artifact(run_dir, "model_inventory.json", model_inventory)

print("Teacher params:", teacher_params)
print("Teacher MACs (rough):", teacher_maccs)
print("Student params:", student_params)
print("Student MACs (rough):", student_maccs)

teacher.summary()
student.summary()


In [ ]:
teacher_reused = False
teacher_checkpoint_path = resolved_teacher_checkpoint if REUSE_TEACHER_CHECKPOINT else None

if teacher_checkpoint_path is not None and Path(teacher_checkpoint_path).exists():
    teacher = tf.keras.models.load_model(teacher_checkpoint_path)
    teacher_reused = True
    teacher_history = None
    teacher_summary_path = save_model_summary(run_dir, teacher, "teacher")
    add_note(
        run_dir,
        "Teacher reuse",
        f"Loaded teacher checkpoint from `{teacher_checkpoint_path}`.",
    )
    print("Loaded teacher from:", teacher_checkpoint_path)
else:
    teacher = compile_classifier(
        teacher,
        experiment=experiment,
        learning_rate=experiment.train.teacher_lr,
        epochs=experiment.train.teacher_epochs,
        steps_per_epoch=steps_per_epoch,
        label_smoothing=experiment.model.teacher_label_smoothing,
        warmup_epochs=experiment.train.teacher_warmup_epochs,
    )
    teacher_history = teacher.fit(
        teacher_train_ds,
        validation_data=teacher_val_ds,
        epochs=experiment.train.teacher_epochs,
        callbacks=build_callbacks_for_monitor(
            teacher_dir,
            patience=experiment.train.teacher_early_stopping_patience,
            monitor="val_accuracy",
            mode="max",
            include_checkpoint=True,
            min_delta=experiment.train.teacher_early_stopping_min_delta,
        ),
    )

teacher_test_metrics = teacher.evaluate(teacher_test_ds, return_dict=True)
teacher_history_path = save_history(run_dir, teacher_history, "teacher")
save_metrics(run_dir, teacher_test_metrics, "teacher")
teacher_plot_paths = save_history_plots(run_dir, teacher_history, "teacher")
update_stage_state(
    run_dir,
    "teacher",
    metrics=teacher_test_metrics,
    history_path=str(teacher_history_path) if teacher_history_path else "",
    plot_paths=teacher_plot_paths,
    summary_path=str(teacher_summary_path),
    extra={
        "params": teacher_params,
        "maccs_rough": teacher_maccs,
        "epoch_log_path": "" if teacher_reused else str(teacher_dir / "epoch_log.csv"),
        "checkpoint_path": (
            str(teacher_checkpoint_path)
            if teacher_reused and teacher_checkpoint_path is not None
            else str(teacher_dir / "best.keras")
        ),
        "reused_from_checkpoint": teacher_reused,
    },
)
print("Teacher test metrics:", teacher_test_metrics)


In [ ]:
student_training = train_student_model(
    student=student,
    teacher=teacher,
    train_ds=distillation_train_ds,
    val_ds=distillation_val_ds,
    experiment=experiment,
    steps_per_epoch=steps_per_epoch,
    output_dir=student_dir,
)
student = student_training["student"]
student_history = student_training["history"]
student_training_extra = student_training["extra"]

student = compile_classifier(
    student,
    experiment=experiment,
    learning_rate=experiment.train.student_lr,
    epochs=1,
    steps_per_epoch=max(1, steps_per_epoch),
)
student_test_metrics = student.evaluate(student_test_ds, return_dict=True)
student_history_path = save_history(run_dir, student_history, "student")
save_metrics(run_dir, student_test_metrics, "student")
student_plot_paths = save_history_plots(run_dir, student_history, "student")
update_stage_state(
    run_dir,
    "student",
    metrics=student_test_metrics,
    history_path=str(student_history_path) if student_history_path else "",
    plot_paths=student_plot_paths,
    summary_path=str(student_summary_path),
    extra={
        "params": student_params,
        "maccs_rough": student_maccs,
        **student_training_extra,
    },
)
print("Student test metrics:", student_test_metrics)


In [ ]:
final_model = student

if experiment.train.use_qat:
    qat_candidate = maybe_quantize_aware_clone(student)
    if qat_candidate is None:
        print("QAT skipped: tensorflow_model_optimization is not available.")
        add_note(
            run_dir,
            "QAT skipped",
            "tensorflow_model_optimization was not available in the notebook runtime.",
        )
    else:
        qat_candidate.set_weights(student.get_weights())
        qat_summary_path = save_model_summary(run_dir, qat_candidate, "qat")
        qat_candidate = compile_classifier(
            qat_candidate,
            experiment=experiment,
            learning_rate=experiment.train.qat_lr,
            epochs=experiment.train.qat_epochs,
            steps_per_epoch=steps_per_epoch,
        )
        qat_history = qat_candidate.fit(
            student_train_ds,
            validation_data=student_val_ds,
            epochs=experiment.train.qat_epochs,
            callbacks=build_callbacks_for_monitor(
                qat_dir,
                patience=max(2, experiment.train.early_stopping_patience // 2),
                monitor="val_accuracy",
                mode="max",
                include_checkpoint=True,
            ),
        )
        qat_test_metrics = qat_candidate.evaluate(student_test_ds, return_dict=True)
        qat_history_path = save_history(run_dir, qat_history, "qat")
        save_metrics(run_dir, qat_test_metrics, "qat")
        qat_plot_paths = save_history_plots(run_dir, qat_history, "qat")
        update_stage_state(
            run_dir,
            "qat",
            metrics=qat_test_metrics,
            history_path=str(qat_history_path) if qat_history_path else "",
            plot_paths=qat_plot_paths,
            summary_path=str(qat_summary_path),
            extra={
                "epoch_log_path": str(qat_dir / "epoch_log.csv"),
                "checkpoint_path": str(qat_dir / "best.keras"),
            },
        )
        print("QAT student test metrics:", qat_test_metrics)
        final_model = qat_candidate


In [ ]:
metadata = export_experiment_artifacts(
    model=final_model,
    representative_data=student_train_ds,
    experiment=experiment,
    required_ops=required_tflite_ops(experiment.model.student_name),
)
record_export_artifacts(run_dir, metadata)
save_json_artifact(run_dir, "export_metadata_snapshot.json", metadata)
print(json.dumps(metadata, indent=2))


In [ ]:
def plot_history(history, title):
    if history is None:
        return
    hist = history.history
    keys = [key for key in hist if "accuracy" in key or key == "loss"]
    if not keys:
        return
    plt.figure(figsize=(12, 4))
    for key in keys:
        plt.plot(hist[key], label=key)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    plt.close()

plot_history(teacher_history, "Teacher history")
plot_history(student_history, "Student history")
if "qat_history" in globals():
    plot_history(qat_history, "QAT history")


In [ ]:
summary_path = write_run_summary(run_dir)
print("Run summary:", summary_path)
print("Teacher epoch log:", teacher_dir / "epoch_log.csv")
print("Student epoch log:", student_dir / "epoch_log.csv")
if experiment.train.use_qat:
    print("QAT epoch log:", qat_dir / "epoch_log.csv")


In [ ]:
import shutil

archive_base = run_dir.parent / f"{run_dir.name}_bundle"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=run_dir)
DRIVE_EXPORT_DIR = ""

print("Run archive:", archive_path)
if DRIVE_EXPORT_DIR:
    drive_export_dir = Path(DRIVE_EXPORT_DIR)
    drive_export_dir.mkdir(parents=True, exist_ok=True)
    exported_archive = drive_export_dir / Path(archive_path).name
    shutil.copy2(archive_path, exported_archive)
    print("Copied archive:", exported_archive)


## Next Iteration Ideas

- Compare `distilled_student_v2` against `distilled_student_v3`, `distilled_student_v3_teacher_plus`, and `distilled_student_v3_bcresnet_teacher`.
- Use `baseline` and `fast_student` only as sanity references, not as the main optimization path.
- Run `kws20_probe` after a stable `kws12` export exists.
- If exact microfrontend import fails, keep the fallback pipeline for rapid experimentation, but verify transfer to firmware before trusting notebook gains.
